# Orbit Wars — cnn_v1 Self-Play PPO (Colab v3)

Changes from v2:
- Per-iter progress adds W/L/D counts, avg episode steps, opponent-type breakdown, total elapsed, ETA, and GPU memory.
- Model is saved **every iter** to latest.pt (separate from best-by-eval cnn_v1.pt).
- Per-episode line shows running W/L/D and iter-elapsed.

Outputs under /content/orbit-wars/:
- agents/cnn_v1/weights/cnn_v1.pt — best weights (gated by win rate vs physical_v2)
- agents/cnn_v1/weights/latest.pt — most recent iter (resume-friendly)
- logs/training/ppo_<timestamp>.jsonl — per-iter metrics (now includes wins, losses, draws, avg_episode_steps, elapsed_s, eta_s, gpu_mem_gb)
- logs/replays/training/<run_id>/*.html — game records every 10 iters


In [ ]:
!pip install -q kaggle-environments>=1.28.0


## Write source files

In [ ]:
import base64, os
from pathlib import Path

BASE = Path('/content/orbit-wars')
BASE.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)

FILES = {
    'agents/registry.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZQoKQWdlbnRGbiA9IENhbGxhYmxlW1tkaWN0XSwgbGlzdF0KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBBZ2VudFNwZWM6CiAgICBpZDogc3RyCiAgICBmbjogQWdlbnRGbgogICAgZGVzY3JpcHRpb246IHN0cgoKCl9SRUdJU1RSWTogZGljdFtzdHIsIEFnZW50U3BlY10gPSB7fQoKCmRlZiByZWdpc3RlcihpZDogc3RyLCBkZXNjcmlwdGlvbjogc3RyKToKICAgIGRlZiBkZWNvcmF0b3IoZm46IEFnZW50Rm4pIC0+IEFnZW50Rm46CiAgICAgICAgaWYgaWQgaW4gX1JFR0lTVFJZOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiYWdlbnQgaWQge2lkIXJ9IGFscmVhZHkgcmVnaXN0ZXJlZCIpCiAgICAgICAgX1JFR0lTVFJZW2lkXSA9IEFnZW50U3BlYyhpZD1pZCwgZm49Zm4sIGRlc2NyaXB0aW9uPWRlc2NyaXB0aW9uKQogICAgICAgIHJldHVybiBmbgoKICAgIHJldHVybiBkZWNvcmF0b3IKCgpkZWYgbGlzdF9hZ2VudHMoKSAtPiBsaXN0W3N0cl06CiAgICByZXR1cm4gc29ydGVkKF9SRUdJU1RSWSkKCgpkZWYgbGlzdF9hZ2VudF9zcGVjcygpIC0+IGxpc3RbQWdlbnRTcGVjXToKICAgIHJldHVybiBbX1JFR0lTVFJZW2tdIGZvciBrIGluIHNvcnRlZChfUkVHSVNUUlkpXQoKCmNsYXNzIEFnZW50OgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlkOiBzdHIpOgogICAgICAgIGlmIGlkIG5vdCBpbiBfUkVHSVNUUlk6CiAgICAgICAgICAgIHJhaXNlIEtleUVycm9yKGYidW5rbm93biBhZ2VudCBpZCB7aWQhcn0uIGF2YWlsYWJsZToge2xpc3RfYWdlbnRzKCl9IikKICAgICAgICBzcGVjID0gX1JFR0lTVFJZW2lkXQogICAgICAgIHNlbGYuaWQgPSBzcGVjLmlkCiAgICAgICAgc2VsZi5mbjogQWdlbnRGbiA9IHNwZWMuZm4KICAgICAgICBzZWxmLmRlc2NyaXB0aW9uID0gc3BlYy5kZXNjcmlwdGlvbgoKICAgIGRlZiBfX2NhbGxfXyhzZWxmLCBvYnMpOgogICAgICAgIHJldHVybiBzZWxmLmZuKG9icykKCiAgICBkZWYgX19yZXByX18oc2VsZikgLT4gc3RyOgogICAgICAgIHJldHVybiBmIkFnZW50KGlkPXtzZWxmLmlkIXJ9KSIK',
    'agents/physical_v2/__init__.py': 'ZnJvbSAuIGltcG9ydCBhZ2VudCAgIyBub3FhOiBGNDAxICDigJQgdHJpZ2dlcnMgQHJlZ2lzdGVyCg==',
    'agents/physical_v2/agent.py': 'IiIiSGV1cmlzdGljIHBoeXNpY2FsIGFnZW50IHYyIOKAlCB2MSArIGRlZmVuc2l2ZSB0aHJlYXQgYWNjb3VudGluZy4KClNpbmdsZSBpbXByb3ZlbWVudCBvdmVyIHYxOiBiZWZvcmUgcGlja2luZyBhIHRhcmdldCBmcm9tIGEgc291cmNlIHBsYW5ldCwKY29tcHV0ZSB0aGUgbWluaW11bSBnYXJyaXNvbiB0aGUgcGxhbmV0IHdpbGwgc2VlIG92ZXIgdGhlIHRpbWVsaW5lIG9mCmluY29taW5nIGVuZW15IGZsZWV0cy4gT25seSB0aGUgKnN1cnBsdXMqIGFib3ZlIHRoYXQgbWluaW11bSAobWludXMgYQpkZWZlbnNpdmUgYnVmZmVyKSBpcyBhdmFpbGFibGUgZm9yIG9mZmVuc2UuIFNvdXJjZXMgdW5kZXIgdGhyZWF0IGhvbGQuCgpUaHJlYXQgbW9kZWwgcGVyIHNvdXJjZToKICAxLiBGb3IgZWFjaCBlbmVteSBmbGVldCBpbiBmbGlnaHQsIGNvbXB1dGUgd2hlbiBpdHMgc3RyYWlnaHQtbGluZQogICAgIHRyYWplY3RvcnkgY29tZXMgd2l0aGluIHRoZSBwbGFuZXQncyByYWRpdXMgKE5vbmUgaWYgaXQgbWlzc2VzIG9yCiAgICAgaXMgbW92aW5nIGF3YXkpLgogIDIuIFNvcnQgdGhyZWF0cyBieSBhcnJpdmFsIHR1cm47IHdhbGsgdGhlIHRpbWVsaW5lIGZvcndhcmQ6CiAgICAgICBnYXJyaXNvbih0KSA9IGdhcnJpc29uKGxhc3RfdCkgKyBwcm9kdWN0aW9uIMK3ICh0IOKIkiBsYXN0X3QpIOKIkiB0aHJlYXRfc2hpcHModCkKICAgICBUcmFjayB0aGUgbWluaW11bSBnYXJyaXNvbiBhY3Jvc3MgYWxsIGFycml2YWxzLgogIDMuIFN1cnBsdXMgPSBtaW5fZ2Fycmlzb24g4oiSIERFRkVOU0VfQlVGRkVSLgoKRXZlcnl0aGluZyBlbHNlIChsZWFkLWFpbSwgc3VuLWRvZGdlLCByb3RhdGlvbiBzaWduIGluZmVyZW5jZSwKdHJhdmVsX3RpbWUvcHJvZHVjdGlvbiBzY29yaW5nLCBwcm9kdWN0aW9uLWR1cmluZy10cmF2ZWwgYWxsb2NhdGlvbikgaXMKY29waWVkIHZlcmJhdGltIGZyb20gdjEgc28gdGhlIGZpbGUgc3RheXMgc2VsZi1jb250YWluZWQgZm9yIHBhY2tpbmcuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKCmZyb20ga2FnZ2xlX2Vudmlyb25tZW50cy5lbnZzLm9yYml0X3dhcnMub3JiaXRfd2FycyBpbXBvcnQgRmxlZXQsIFBsYW5ldAoKZnJvbSAuLnJlZ2lzdHJ5IGltcG9ydCByZWdpc3RlcgoKU1VOX0NYID0gNTAuMApTVU5fQ1kgPSA1MC4wClNVTl9SQURJVVMgPSAxMC4wClNVTl9NQVJHSU4gPSAxLjAKTUFYX1NQRUVEID0gNi4wClNQRUVEX0xPR19ERU5PTSA9IG1hdGgubG9nKDEwMDAuMCkKUk9UQVRJT05fUkFESVVTX0xJTUlUID0gNTAuMApMRUFEX0FJTV9JVEVSUyA9IDYKU0FGRVRZX0JVRkZFUiA9IDMKTUlOX0xBVU5DSF9TSElQUyA9IDUKTkVVVFJBTF9CT05VUyA9IDAuOApERUZFTlNFX0JVRkZFUiA9IDMKCgpkZWYgZmxlZXRfc3BlZWQoc2hpcHM6IGludCkgLT4gZmxvYXQ6CiAgICBpZiBzaGlwcyA8PSAxOgogICAgICAgIHJldHVybiAxLjAKICAgIHJldHVybiAxLjAgKyAoTUFYX1NQRUVEIC0gMS4wKSAqIChtYXRoLmxvZyhzaGlwcykgLyBTUEVFRF9MT0dfREVOT00pICoqIDEuNQoKCmRlZiBfZGlzdF9mcm9tX3N1bih4OiBmbG9hdCwgeTogZmxvYXQpIC0+IGZsb2F0OgogICAgcmV0dXJuIG1hdGguaHlwb3QoeCAtIFNVTl9DWCwgeSAtIFNVTl9DWSkKCgpkZWYgaXNfb3JiaXRpbmcocDogUGxhbmV0LCBhbmd1bGFyX3ZlbG9jaXR5OiBmbG9hdCkgLT4gYm9vbDoKICAgIGlmIGFuZ3VsYXJfdmVsb2NpdHkgPT0gMDoKICAgICAgICByZXR1cm4gRmFsc2UKICAgIHJldHVybiAoX2Rpc3RfZnJvbV9zdW4ocC54LCBwLnkpICsgcC5yYWRpdXMpIDwgUk9UQVRJT05fUkFESVVTX0xJTUlUCgoKZGVmIGNyb3NzZXNfc3VuKHgxOiBmbG9hdCwgeTE6IGZsb2F0LCB4MjogZmxvYXQsIHkyOiBmbG9hdCkgLT4gYm9vbDoKICAgIGR4LCBkeSA9IHgyIC0geDEsIHkyIC0geTEKICAgIGxlbl9zcSA9IGR4ICogZHggKyBkeSAqIGR5CiAgICBpZiBsZW5fc3EgPT0gMDoKICAgICAgICByZXR1cm4gX2Rpc3RfZnJvbV9zdW4oeDEsIHkxKSA8PSBTVU5fUkFESVVTICsgU1VOX01BUkdJTgogICAgdCA9IG1heCgwLjAsIG1pbigxLjAsICgoU1VOX0NYIC0geDEpICogZHggKyAoU1VOX0NZIC0geTEpICogZHkpIC8gbGVuX3NxKSkKICAgIGN4LCBjeSA9IHgxICsgdCAqIGR4LCB5MSArIHQgKiBkeQogICAgcmV0dXJuIG1hdGguaHlwb3QoY3ggLSBTVU5fQ1gsIGN5IC0gU1VOX0NZKSA8PSBTVU5fUkFESVVTICsgU1VOX01BUkdJTgoKCmRlZiBpbmZlcl9yb3RhdGlvbl9zaWduKHBsYW5ldHM6IGxpc3RbUGxhbmV0XSwgaW5pdGlhbF9wbGFuZXRzOiBsaXN0KSAtPiBpbnQ6CiAgICBpbml0ID0ge3Jvd1swXTogcm93IGZvciByb3cgaW4gaW5pdGlhbF9wbGFuZXRzfQogICAgZm9yIHAgaW4gcGxhbmV0czoKICAgICAgICBpZiBwLmlkIG5vdCBpbiBpbml0OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlwID0gaW5pdFtwLmlkXQogICAgICAgIGl4LCBpeSA9IGlwWzJdLCBpcFszXQogICAgICAgIGlyID0gbWF0aC5oeXBvdChpeCAtIFNVTl9DWCwgaXkgLSBTVU5fQ1kpCiAgICAgICAgY3IgPSBfZGlzdF9mcm9tX3N1bihwLngsIHAueSkKICAgICAgICBpZiBhYnMoaXIgLSBjcikgPiAwLjU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWEgPSBtYXRoLmF0YW4yKGl5IC0gU1VOX0NZLCBpeCAtIFNVTl9DWCkKICAgICAgICBjYSA9IG1hdGguYXRhbjIocC55IC0gU1VOX0NZLCBwLnggLSBTVU5fQ1gpCiAgICAgICAgZGVsdGEgPSAoY2EgLSBpYSArIG1hdGgucGkpICUgKDIgKiBtYXRoLnBpKSAtIG1hdGgucGkKICAgICAgICBpZiBhYnMoZGVsdGEpID4gMWUtMzoKICAgICAgICAgICAgcmV0dXJuIDEgaWYgZGVsdGEgPiAwIGVsc2UgLTEKICAgIHJldHVybiAxCgoKZGVmIHByZWRpY3RfcG9zaXRpb24ocDogUGxhbmV0LCBhdl9zaWduZWQ6IGZsb2F0LCB0dXJuczogZmxvYXQpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdF06CiAgICBkeCwgZHkgPSBwLnggLSBTVU5fQ1gsIHAueSAtIFNVTl9DWQogICAgciA9IG1hdGguaHlwb3QoZHgsIGR5KQogICAgYW5nbGUgPSBtYXRoLmF0YW4yKGR5LCBkeCkgKyBhdl9zaWduZWQgKiB0dXJucwogICAgcmV0dXJuIFNVTl9DWCArIHIgKiBtYXRoLmNvcyhhbmdsZSksIFNVTl9DWSArIHIgKiBtYXRoLnNpbihhbmdsZSkKCgpkZWYgbGVhZF9haW0oCiAgICBzb3VyY2U6IFBsYW5ldCwKICAgIHRhcmdldDogUGxhbmV0LAogICAgZmxlZXRfc2hpcHM6IGludCwKICAgIGF2X3NpZ25lZDogZmxvYXQsCiAgICBvcmJpdGluZzogYm9vbCwKKSAtPiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XToKICAgIHNwZWVkID0gZmxlZXRfc3BlZWQoZmxlZXRfc2hpcHMpCiAgICBpZiBub3Qgb3JiaXRpbmc6CiAgICAgICAgZGlzdCA9IG1hdGguaHlwb3QodGFyZ2V0LnggLSBzb3VyY2UueCwgdGFyZ2V0LnkgLSBzb3VyY2UueSkKICAgICAgICByZXR1cm4gdGFyZ2V0LngsIHRhcmdldC55LCBkaXN0IC8gc3BlZWQKICAgIHB4LCBweSA9IHRhcmdldC54LCB0YXJnZXQueQogICAgdHVybnMgPSBtYXRoLmh5cG90KHB4IC0gc291cmNlLngsIHB5IC0gc291cmNlLnkpIC8gc3BlZWQKICAgIGZvciBfIGluIHJhbmdlKExFQURfQUlNX0lURVJTKToKICAgICAgICBweCwgcHkgPSBwcmVkaWN0X3Bvc2l0aW9uKHRhcmdldCwgYXZfc2lnbmVkLCB0dXJucykKICAgICAgICB0dXJucyA9IG1hdGguaHlwb3QocHggLSBzb3VyY2UueCwgcHkgLSBzb3VyY2UueSkgLyBzcGVlZAogICAgcmV0dXJuIHB4LCBweSwgdHVybnMKCgpkZWYgZmxlZXRfZXRhX3RvX3BsYW5ldChmbGVldDogRmxlZXQsIHBsYW5ldDogUGxhbmV0KSAtPiBmbG9hdCB8IE5vbmU6CiAgICAiIiJUaW1lIHVudGlsIGBgZmxlZXRgYCdzIHRyYWplY3RvcnkgY29tZXMgd2l0aGluIGBgcGxhbmV0LnJhZGl1c2BgLgoKICAgIFJldHVybnMgTm9uZSBpZiB0aGUgZmxlZXQgbWlzc2VzIHRoZSBwbGFuZXQgb3IgaXMgbW92aW5nIGF3YXkuCiAgICBUcmVhdHMgdGhlIHBsYW5ldCBhcyBzdGF0aWMg4oCUIGdvb2QgZW5vdWdoIGZvciBhIHYyIHRocmVhdCBoZXVyaXN0aWM7CiAgICBvcmJpdGluZyBwbGFuZXRzIHRoYXQgbW92ZSBhd2F5IGNhbiBvbmx5IG92ZXItY291bnQgdGhyZWF0LCB3aGljaAogICAgdGhlIERFRkVOU0VfQlVGRkVSIGFscmVhZHkgYWJzb3Jicy4KICAgICIiIgogICAgc3BlZWQgPSBmbGVldF9zcGVlZChmbGVldC5zaGlwcykKICAgIGlmIHNwZWVkIDw9IDA6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGNoID0gbWF0aC5jb3MoZmxlZXQuYW5nbGUpCiAgICBzaCA9IG1hdGguc2luKGZsZWV0LmFuZ2xlKQogICAgZHggPSBwbGFuZXQueCAtIGZsZWV0LngKICAgIGR5ID0gcGxhbmV0LnkgLSBmbGVldC55CiAgICB0X2Nsb3Nlc3QgPSAoZHggKiBjaCArIGR5ICogc2gpIC8gc3BlZWQKICAgIGlmIHRfY2xvc2VzdCA8IDA6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGN4ID0gZmxlZXQueCArIHNwZWVkICogdF9jbG9zZXN0ICogY2gKICAgIGN5ID0gZmxlZXQueSArIHNwZWVkICogdF9jbG9zZXN0ICogc2gKICAgIGlmIG1hdGguaHlwb3QoY3ggLSBwbGFuZXQueCwgY3kgLSBwbGFuZXQueSkgPiBwbGFuZXQucmFkaXVzOgogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gdF9jbG9zZXN0CgoKZGVmIGNvbXB1dGVfc3VycGx1cyhzb3VyY2U6IFBsYW5ldCwgZW5lbXlfZmxlZXRzOiBsaXN0W0ZsZWV0XSkgLT4gaW50OgogICAgIiIiTWluIGdhcnJpc29uIG92ZXIgdGhlIHRocmVhdCB0aW1lbGluZSwgbWludXMgREVGRU5TRV9CVUZGRVIuIiIiCiAgICB0aHJlYXRzID0gW10KICAgIGZvciBmIGluIGVuZW15X2ZsZWV0czoKICAgICAgICBldGEgPSBmbGVldF9ldGFfdG9fcGxhbmV0KGYsIHNvdXJjZSkKICAgICAgICBpZiBldGEgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRocmVhdHMuYXBwZW5kKChldGEsIGYuc2hpcHMpKQogICAgaWYgbm90IHRocmVhdHM6CiAgICAgICAgcmV0dXJuIG1heCgwLCBzb3VyY2Uuc2hpcHMgLSBERUZFTlNFX0JVRkZFUikKICAgIHRocmVhdHMuc29ydChrZXk9bGFtYmRhIHg6IHhbMF0pCiAgICBnYXJyaXNvbiA9IGZsb2F0KHNvdXJjZS5zaGlwcykKICAgIG1pbl9nYXJyaXNvbiA9IGdhcnJpc29uCiAgICBsYXN0X3QgPSAwLjAKICAgIGZvciB0LCBzaGlwcyBpbiB0aHJlYXRzOgogICAgICAgIGdhcnJpc29uICs9IHNvdXJjZS5wcm9kdWN0aW9uICogKHQgLSBsYXN0X3QpCiAgICAgICAgZ2Fycmlzb24gLT0gc2hpcHMKICAgICAgICBpZiBnYXJyaXNvbiA8IG1pbl9nYXJyaXNvbjoKICAgICAgICAgICAgbWluX2dhcnJpc29uID0gZ2Fycmlzb24KICAgICAgICBsYXN0X3QgPSB0CiAgICByZXR1cm4gaW50KG1heCgwLCBtYXRoLmZsb29yKG1pbl9nYXJyaXNvbikgLSBERUZFTlNFX0JVRkZFUikpCgoKZGVmIF9zY29yZSh0dXJuczogZmxvYXQsIHRhcmdldDogUGxhbmV0KSAtPiBmbG9hdDoKICAgIGJhc2UgPSB0dXJucyAvIG1heCgxLCB0YXJnZXQucHJvZHVjdGlvbikKICAgIHJldHVybiBiYXNlICogKE5FVVRSQUxfQk9OVVMgaWYgdGFyZ2V0Lm93bmVyID09IC0xIGVsc2UgMS4wKQoKCkByZWdpc3RlcigKICAgICJwaHlzaWNhbF92MiIsCiAgICAicGh5c2ljYWxfdjEgKyBkZWZlbnNpdmUgdGhyZWF0IGFjY291bnRpbmcuIEluY29taW5nIGVuZW15IGZsZWV0IHRyYWplY3RvcmllcyAiCiAgICAiY29uc3RyYWluIHRoZSBsYXVuY2ggYnVkZ2V0IHBlciBwbGFuZXQgc28gdGhyZWF0ZW5lZCBzb3VyY2VzIGhvbGQuIiwKKQpkZWYgcGh5c2ljYWxfdjJfYWdlbnQob2JzKToKICAgIHBsYXllciA9IG9icy5nZXQoInBsYXllciIsIDApIGlmIGlzaW5zdGFuY2Uob2JzLCBkaWN0KSBlbHNlIG9icy5wbGF5ZXIKICAgIGdldCA9IG9icy5nZXQgaWYgaXNpbnN0YW5jZShvYnMsIGRpY3QpIGVsc2UgbGFtYmRhIGssIGQ9Tm9uZTogZ2V0YXR0cihvYnMsIGssIGQpCiAgICByYXdfcGxhbmV0cyA9IGdldCgicGxhbmV0cyIpIG9yIFtdCiAgICByYXdfZmxlZXRzID0gZ2V0KCJmbGVldHMiKSBvciBbXQogICAgYW5ndWxhcl92ZWxvY2l0eSA9IGFicyhmbG9hdChnZXQoImFuZ3VsYXJfdmVsb2NpdHkiKSBvciAwLjApKQogICAgaW5pdGlhbF9wbGFuZXRzID0gZ2V0KCJpbml0aWFsX3BsYW5ldHMiKSBvciBbXQoKICAgIHBsYW5ldHMgPSBbUGxhbmV0KCpwKSBmb3IgcCBpbiByYXdfcGxhbmV0c10KICAgIGZsZWV0cyA9IFtGbGVldCgqZikgZm9yIGYgaW4gcmF3X2ZsZWV0c10KICAgIGF2X3NpZ24gPSBpbmZlcl9yb3RhdGlvbl9zaWduKHBsYW5ldHMsIGluaXRpYWxfcGxhbmV0cykKICAgIGF2X3NpZ25lZCA9IGFuZ3VsYXJfdmVsb2NpdHkgKiBhdl9zaWduCgogICAgbXlfcGxhbmV0cyA9IFtwIGZvciBwIGluIHBsYW5ldHMgaWYgcC5vd25lciA9PSBwbGF5ZXIgYW5kIHAuc2hpcHMgPj0gTUlOX0xBVU5DSF9TSElQU10KICAgIHRhcmdldHMgPSBbcCBmb3IgcCBpbiBwbGFuZXRzIGlmIHAub3duZXIgIT0gcGxheWVyXQogICAgZW5lbXlfZmxlZXRzID0gW2YgZm9yIGYgaW4gZmxlZXRzIGlmIGYub3duZXIgIT0gcGxheWVyIGFuZCBmLm93bmVyID49IDBdCiAgICBpZiBub3QgbXlfcGxhbmV0cyBvciBub3QgdGFyZ2V0czoKICAgICAgICByZXR1cm4gW10KCiAgICBtb3ZlcyA9IFtdCiAgICBmb3Igc291cmNlIGluIG15X3BsYW5ldHM6CiAgICAgICAgc3VycGx1cyA9IGNvbXB1dGVfc3VycGx1cyhzb3VyY2UsIGVuZW15X2ZsZWV0cykKICAgICAgICBpZiBzdXJwbHVzIDwgTUlOX0xBVU5DSF9TSElQUzoKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgYmVzdCA9IE5vbmUKICAgICAgICBiZXN0X3Njb3JlID0gZmxvYXQoImluZiIpCiAgICAgICAgYmVzdF9hbmdsZSA9IDAuMAogICAgICAgIGJlc3Rfc2hpcHMgPSAwCgogICAgICAgIGZvciB0YXJnZXQgaW4gdGFyZ2V0czoKICAgICAgICAgICAgb3JiaXRpbmcgPSBpc19vcmJpdGluZyh0YXJnZXQsIGFuZ3VsYXJfdmVsb2NpdHkpCiAgICAgICAgICAgIGZsZWV0X2d1ZXNzID0gbWluKG1heCh0YXJnZXQuc2hpcHMgKyAxMCwgMjApLCBzdXJwbHVzKQogICAgICAgICAgICBpZiBmbGVldF9ndWVzcyA8IDE6CiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgcHgsIHB5LCB0dXJucyA9IGxlYWRfYWltKHNvdXJjZSwgdGFyZ2V0LCBmbGVldF9ndWVzcywgYXZfc2lnbmVkLCBvcmJpdGluZykKICAgICAgICAgICAgaWYgY3Jvc3Nlc19zdW4oc291cmNlLngsIHNvdXJjZS55LCBweCwgcHkpOgogICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgICAgIGlmIHRhcmdldC5vd25lciA9PSAtMToKICAgICAgICAgICAgICAgIHNoaXBzX29uX2Fycml2YWwgPSB0YXJnZXQuc2hpcHMKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHNoaXBzX29uX2Fycml2YWwgPSB0YXJnZXQuc2hpcHMgKyBpbnQodGFyZ2V0LnByb2R1Y3Rpb24gKiB0dXJucykKICAgICAgICAgICAgc2hpcHNfbmVlZGVkID0gc2hpcHNfb25fYXJyaXZhbCArIFNBRkVUWV9CVUZGRVIKCiAgICAgICAgICAgIGlmIHNoaXBzX25lZWRlZCA+IHN1cnBsdXM6CiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgc2MgPSBfc2NvcmUodHVybnMsIHRhcmdldCkKICAgICAgICAgICAgaWYgc2MgPCBiZXN0X3Njb3JlOgogICAgICAgICAgICAgICAgYmVzdF9zY29yZSA9IHNjCiAgICAgICAgICAgICAgICBiZXN0ID0gdGFyZ2V0CiAgICAgICAgICAgICAgICBiZXN0X2FuZ2xlID0gbWF0aC5hdGFuMihweSAtIHNvdXJjZS55LCBweCAtIHNvdXJjZS54KQogICAgICAgICAgICAgICAgYmVzdF9zaGlwcyA9IHNoaXBzX25lZWRlZAoKICAgICAgICBpZiBiZXN0IGlzIG5vdCBOb25lIGFuZCBiZXN0X3NoaXBzID4gMDoKICAgICAgICAgICAgbW92ZXMuYXBwZW5kKFtzb3VyY2UuaWQsIGJlc3RfYW5nbGUsIGJlc3Rfc2hpcHNdKQoKICAgIHJldHVybiBtb3Zlcwo=',
    'agents/cnn_v1/__init__.py': 'IiIiQ05OIHYxIGFnZW50IOKAlCBtb2RlbCwgdHJhaW5pbmcgcGlwZWxpbmVzLCBldmFsIGFsbCBsaXZlIGhlcmUuIiIiCgpmcm9tIC5hZ2VudCBpbXBvcnQgKAogICAgQk9BUkRfU0laRSwKICAgIENFTEwsCiAgICBDTk52MSwKICAgIERfTU9ERUwsCiAgICBHUklELAogICAgTEFVTkNIX1RIUkVTSE9MRCwKICAgIE1BWF9TVEVQUywKICAgIE5VTV9DSEFOTkVMUywKICAgIFNDQUxBUl9ESU0sCiAgICBTSElQU19MT0dfTUFYLAogICAgV0VJR0hUU19QQVRILAogICAgY25uX3YxX2FnZW50LAogICAgZmVhdHVyaXplLAogICAgcmVsb2FkX3dlaWdodHMsCikKZnJvbSAuYmMgaW1wb3J0IHRyYWluX2JjCmZyb20gLmNvbGxlY3QgaW1wb3J0IGNvbGxlY3RfYmNfZGF0YXNldCwgZGVmYXVsdF9kYXRhc2V0X3BhdGgKZnJvbSAuY29tbW9uIGltcG9ydCAoCiAgICBhbmdsZV90b19jZWxsLAogICAgZGVjb2RlX3RlYWNoZXJfYWN0aW9uLAogICAgZnJlc2hfbW9kZWwsCiAgICBsb2FkX21vZGVsLAogICAgc2F2ZV9tb2RlbCwKKQpmcm9tIC5ldmFsIGltcG9ydCBldmFsdWF0ZV9hZ2VudApmcm9tIC5wcG8gaW1wb3J0IHRyYWluX3BwbwoKX19hbGxfXyA9IFsKICAgICJCT0FSRF9TSVpFIiwKICAgICJDRUxMIiwKICAgICJDTk52MSIsCiAgICAiRF9NT0RFTCIsCiAgICAiR1JJRCIsCiAgICAiTEFVTkNIX1RIUkVTSE9MRCIsCiAgICAiTUFYX1NURVBTIiwKICAgICJOVU1fQ0hBTk5FTFMiLAogICAgIlNDQUxBUl9ESU0iLAogICAgIlNISVBTX0xPR19NQVgiLAogICAgIldFSUdIVFNfUEFUSCIsCiAgICAiYW5nbGVfdG9fY2VsbCIsCiAgICAiY25uX3YxX2FnZW50IiwKICAgICJjb2xsZWN0X2JjX2RhdGFzZXQiLAogICAgImRlY29kZV90ZWFjaGVyX2FjdGlvbiIsCiAgICAiZGVmYXVsdF9kYXRhc2V0X3BhdGgiLAogICAgImV2YWx1YXRlX2FnZW50IiwKICAgICJmZWF0dXJpemUiLAogICAgImZyZXNoX21vZGVsIiwKICAgICJsb2FkX21vZGVsIiwKICAgICJyZWxvYWRfd2VpZ2h0cyIsCiAgICAic2F2ZV9tb2RlbCIsCiAgICAidHJhaW5fYmMiLAogICAgInRyYWluX3BwbyIsCl0K',
    'agents/cnn_v1/agent.py': 'IiIiQ05OIHYxIGFnZW50IOKAlCBjb252b2x1dGlvbmFsIHBvbGljeSBvdmVyIGEgcGxheWVyLXJlbGF0aXZlIDUww5c1MCByYXN0ZXIuCgotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpJbnB1dCBjaGFubmVscyAoc2hhcGU6IDE1IMOXIDUwIMOXIDUwLCBjZWxsID0gMiB3b3JsZCB1bml0cykKLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KfCBpZHggfCBjaGFubmVsICAgICAgICAgICAgICAgICAgICB8IHdoYXQgaXQgZW5jb2RlcyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHwKfC0tLS0tfC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS18LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLXwKfCAgMCAgfCBteV9wbGFuZXRfc2hpcHMgICAgICAgICAgICB8IGxvZzFwKHNoaXBzKS9sb2coNTAwMCkgYXQgbXkgcGxhbmV0cyAgICAgICAgIHwKfCAgMSAgfCBlbmVteV9wbGFuZXRfc2hpcHMgICAgICAgICB8IC4uLiBhdCBlbmVteSBwbGFuZXRzICAgICAgICAgICAgICAgICAgICAgICAgIHwKfCAgMiAgfCBuZXV0cmFsX3BsYW5ldF9zaGlwcyAgICAgICB8IC4uLiBhdCBuZXV0cmFsIHBsYW5ldHMgICAgICAgICAgICAgICAgICAgICAgIHwKfCAgMyAgfCBteV9wbGFuZXRfcHJvZHVjdGlvbiAgICAgICB8IHByb2R1Y3Rpb24vNSBhdCBteSBwbGFuZXRzICAgICAgICAgICAgICAgICAgIHwKfCAgNCAgfCBlbmVteV9wbGFuZXRfcHJvZHVjdGlvbiAgICB8IC4uLiBhdCBlbmVteSBwbGFuZXRzICAgICAgICAgICAgICAgICAgICAgICAgIHwKfCAgNSAgfCBuZXV0cmFsX3BsYW5ldF9wcm9kdWN0aW9uICB8IC4uLiBhdCBuZXV0cmFsIHBsYW5ldHMgICAgICAgICAgICAgICAgICAgICAgIHwKfCAgNiAgfCBteV9mbGVldF9zaGlwcyAgICAgICAgICAgICB8IGxvZzFwKHNoaXBzKS9sb2coNTAwMCkgYXQgbXkgZmxlZXRzICAgICAgICAgIHwKfCAgNyAgfCBlbmVteV9mbGVldF9zaGlwcyAgICAgICAgICB8IC4uLiBhdCBlbmVteSBmbGVldHMgICAgICAgICAgICAgICAgICAgICAgICAgIHwKfCAgOCAgfCBteV9mbGVldF9zaW4gICAgICAgICAgICAgICB8IHNpbihoZWFkaW5nKSBhdCBteSBmbGVldHMgICAgICAgICAgICAgICAgICAgIHwKfCAgOSAgfCBteV9mbGVldF9jb3MgICAgICAgICAgICAgICB8IGNvcyhoZWFkaW5nKSBhdCBteSBmbGVldHMgICAgICAgICAgICAgICAgICAgIHwKfCAxMCAgfCBlbmVteV9mbGVldF9zaW4gICAgICAgICAgICB8IHNpbihoZWFkaW5nKSBhdCBlbmVteSBmbGVldHMgICAgICAgICAgICAgICAgIHwKfCAxMSAgfCBlbmVteV9mbGVldF9jb3MgICAgICAgICAgICB8IGNvcyhoZWFkaW5nKSBhdCBlbmVteSBmbGVldHMgICAgICAgICAgICAgICAgIHwKfCAxMiAgfCBvcmJpdF9tYXNrICAgICAgICAgICAgICAgICB8IDEgb24gY2VsbHMgY29udGFpbmluZyBhbiBvcmJpdGluZyBwbGFuZXQgICAgIHwKfCAxMyAgfCBjb21ldF9tYXNrICAgICAgICAgICAgICAgICB8IDEgb24gY2VsbHMgY29udGFpbmluZyBhIGNvbWV0ICAgICAgICAgICAgICAgIHwKfCAxNCAgfCBzdW5fbWFzayAgICAgICAgICAgICAgICAgICB8IHN0YXRpYyBkaXNjIGF0IGdyaWQgY2VsbCAoMjUsIDI1KSwgcmFkaXVzIDUgIHwKClNjYWxhciBzaWRlLWNoYW5uZWwgKDcgZmxvYXRzLCBjb25jYXRlbmF0ZWQgYWZ0ZXIgdGhlIGJhY2tib25lKToKICBzdGVwLzUwMCwgbG9nMXAobXlfc2hpcHMpL2xvZzVrLCBsb2cxcChlbmVteV9zaGlwcykvbG9nNWssCiAgbXlfcGxhbmV0X2NvdW50LzEwLCBlbmVteV9wbGFuZXRfY291bnQvMTAsIGFuZ3VsYXJfdmVsb2NpdHksCiAgYWN0aXZlX2NvbWV0X2NvdW50LzIwLgoKLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KQmFja2JvbmUgKHJlY2VwdGl2ZSBmaWVsZCDiiYggMzMgY2VsbHMg4omIIDY2IHdvcmxkIHVuaXRzKQotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogIENvbnYoMTUg4oaSIDMyLCAzw5czKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBSZUxVICBSRj0zCiAgQ29udigzMiDihpIgNjQsIDPDlzMpICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIFJlTFUgIFJGPTUKICBDb252KDY0IOKGkiA2NCwgM8OXMywgZGlsYXRpb249MikgICAgICAgICAgICAgICAgICAgUmVMVSAgUkY9OQogIENvbnYoNjQg4oaSIDY0LCAzw5czLCBkaWxhdGlvbj00KSAgICAgICAgICAgICAgICAgICBSZUxVICBSRj0xNwogIENvbnYoNjQg4oaSIDY0LCAzw5czLCBkaWxhdGlvbj04KSAgICAgICAgICAgICAgICAgICBSZUxVICBSRj0zMwoKU2NhbGFyIE1MUDogNyDihpIgMzIg4oaSIDMyIChSZUxVKS4KCi0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCkFjdGlvbiBoZWFkIChwZXIgb3duZWQgcGxhbmV0KQotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogIGJpbGluZWFyLXNhbXBsZSBmZWF0dXJlIG1hcCBhdCBwbGFuZXQgKHgsIHkpIOKGkiBjb25jYXQgd2l0aCBzY2FsYXIgTUxQCiAgb3V0cHV0IOKGkiBNTFAoOTYg4oaSIDk2KSDihpIgdGhyZWUgaGVhZHM6CgogICAgbGF1bmNoX2xvZ2l0ICAgICAg4oiIIOKEnSAgICAgICAgICAgKHNpZ21vaWQg4oaSIGxhdW5jaCBnYXRlKQogICAgdGFyZ2V0X2xvZ2l0cyAgICAg4oiIIOKEnV4oNTDDlzUwKSAgIChhcmdtYXgg4oaSIHRhcmdldCBjZWxsKQogICAgc2hpcF9mcmFjdGlvbl9sZyAg4oiIIOKEnSAgICAgICAgICAgKHNpZ21vaWQg4oaSIHNoaXAgZnJhY3Rpb24gb2YgZ2Fycmlzb24pCgpUaGUgdGFyZ2V0IGNlbGwg4oaSIGFuZ2xlIHZpYSBhdGFuMih0eSDiiJIgcHksIHR4IOKIkiBweCk7IHNoaXBzID0Kcm91bmQoZnJhY3Rpb24gw5cgZ2Fycmlzb24pLCBjbGlwcGVkIHRvIFsxLCBnYXJyaXNvbl0uCgpTdGF0dXM6IHVudHJhaW5lZC4gRGVmYXVsdCB3ZWlnaHRzIOKGkiBuZWFyLXJhbmRvbSB2YWxpZCBhY3Rpb25zLiBXaGVuCldFSUdIVFNfUEFUSCBleGlzdHMsIGl0IGxvYWRzIG9uIGZpcnN0IGNhbGwuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKaW1wb3J0IHdhcm5pbmdzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgpmcm9tIC4ucmVnaXN0cnkgaW1wb3J0IHJlZ2lzdGVyCgpCT0FSRF9TSVpFID0gMTAwLjAKR1JJRCA9IDUwCkNFTEwgPSBCT0FSRF9TSVpFIC8gR1JJRApOVU1fQ0hBTk5FTFMgPSAxNQpTQ0FMQVJfRElNID0gNwpEX01PREVMID0gNjQKTUFYX1NURVBTID0gNTAwLjAKU0hJUFNfTE9HX01BWCA9IG1hdGgubG9nKDUwMDAuMCkKTEFVTkNIX1RIUkVTSE9MRCA9IDAuNQoKV0VJR0hUU19QQVRIID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCAvICJ3ZWlnaHRzIiAvICJjbm5fdjEucHQiCgoKZGVmIF9jZWxsKHY6IGZsb2F0KSAtPiBpbnQ6CiAgICByZXR1cm4gbWF4KDAsIG1pbihHUklEIC0gMSwgaW50KHYgLyBDRUxMKSkpCgoKZGVmIGZlYXR1cml6ZShvYnMpOgogICAgcGxheWVyID0gb2JzLmdldCgicGxheWVyIiwgMCkgaWYgaXNpbnN0YW5jZShvYnMsIGRpY3QpIGVsc2Ugb2JzLnBsYXllcgogICAgZ2V0ID0gb2JzLmdldCBpZiBpc2luc3RhbmNlKG9icywgZGljdCkgZWxzZSBsYW1iZGEgaywgZD1Ob25lOiBnZXRhdHRyKG9icywgaywgZCkKICAgIHJhd19wbGFuZXRzID0gZ2V0KCJwbGFuZXRzIikgb3IgW10KICAgIHJhd19mbGVldHMgPSBnZXQoImZsZWV0cyIpIG9yIFtdCiAgICBjb21ldF9pZHMgPSBzZXQoZ2V0KCJjb21ldF9wbGFuZXRfaWRzIikgb3IgW10pCiAgICBhbmd1bGFyX3ZlbG9jaXR5ID0gZmxvYXQoZ2V0KCJhbmd1bGFyX3ZlbG9jaXR5Iikgb3IgMC4wKQogICAgc3RlcCA9IGludChnZXQoInN0ZXAiLCAwKSBvciAwKQoKICAgIGNoID0gdG9yY2guemVyb3MoTlVNX0NIQU5ORUxTLCBHUklELCBHUklELCBkdHlwZT10b3JjaC5mbG9hdDMyKQoKICAgIHN5LCBzeCA9IHRvcmNoLm1lc2hncmlkKHRvcmNoLmFyYW5nZShHUklEKSwgdG9yY2guYXJhbmdlKEdSSUQpLCBpbmRleGluZz0iaWoiKQogICAgZGlzdCA9ICgoc3ggLSBHUklEIC8gMikgKiogMiArIChzeSAtIEdSSUQgLyAyKSAqKiAyKS5zcXJ0KCkKICAgIGNoWzE0XSA9IChkaXN0IDw9IDUpLmZsb2F0KCkKCiAgICBteV9zaGlwcyA9IDAKICAgIGVuZW15X3NoaXBzID0gMAogICAgbXlfcGxhbmV0X2NvdW50ID0gMAogICAgZW5lbXlfcGxhbmV0X2NvdW50ID0gMAogICAgbXlfcGxhbmV0cyA9IFtdCgogICAgZm9yIHAgaW4gcmF3X3BsYW5ldHM6CiAgICAgICAgcGlkLCBvd25lciwgeCwgeSwgcmFkaXVzLCBzaGlwcywgcHJvZCA9IHAKICAgICAgICBjeCwgY3kgPSBfY2VsbCh4KSwgX2NlbGwoeSkKICAgICAgICBsb2dfc2hpcHMgPSBtYXRoLmxvZzFwKHNoaXBzKSAvIFNISVBTX0xPR19NQVgKICAgICAgICBwcm9kX25vcm0gPSBwcm9kIC8gNS4wCiAgICAgICAgaXNfb3JiaXRpbmcgPSBhbmd1bGFyX3ZlbG9jaXR5ID4gMCBhbmQgbWF0aC5oeXBvdCh4IC0gNTAsIHkgLSA1MCkgKyByYWRpdXMgPCA1MAogICAgICAgIGlmIG93bmVyID09IHBsYXllcjoKICAgICAgICAgICAgY2hbMCwgY3ksIGN4XSArPSBsb2dfc2hpcHMKICAgICAgICAgICAgY2hbMywgY3ksIGN4XSArPSBwcm9kX25vcm0KICAgICAgICAgICAgbXlfc2hpcHMgKz0gc2hpcHMKICAgICAgICAgICAgbXlfcGxhbmV0X2NvdW50ICs9IDEKICAgICAgICAgICAgbXlfcGxhbmV0cy5hcHBlbmQoKHBpZCwgeCwgeSwgc2hpcHMpKQogICAgICAgIGVsaWYgb3duZXIgPT0gLTE6CiAgICAgICAgICAgIGNoWzIsIGN5LCBjeF0gKz0gbG9nX3NoaXBzCiAgICAgICAgICAgIGNoWzUsIGN5LCBjeF0gKz0gcHJvZF9ub3JtCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY2hbMSwgY3ksIGN4XSArPSBsb2dfc2hpcHMKICAgICAgICAgICAgY2hbNCwgY3ksIGN4XSArPSBwcm9kX25vcm0KICAgICAgICAgICAgZW5lbXlfc2hpcHMgKz0gc2hpcHMKICAgICAgICAgICAgZW5lbXlfcGxhbmV0X2NvdW50ICs9IDEKICAgICAgICBpZiBpc19vcmJpdGluZzoKICAgICAgICAgICAgY2hbMTIsIGN5LCBjeF0gPSAxLjAKICAgICAgICBpZiBwaWQgaW4gY29tZXRfaWRzOgogICAgICAgICAgICBjaFsxMywgY3ksIGN4XSA9IDEuMAoKICAgIGZvciBmIGluIHJhd19mbGVldHM6CiAgICAgICAgZmlkLCBvd25lciwgeCwgeSwgYW5nbGUsIGZyb21fcGlkLCBzaGlwcyA9IGYKICAgICAgICBjeCwgY3kgPSBfY2VsbCh4KSwgX2NlbGwoeSkKICAgICAgICBsb2dfc2hpcHMgPSBtYXRoLmxvZzFwKHNoaXBzKSAvIFNISVBTX0xPR19NQVgKICAgICAgICBpZiBvd25lciA9PSBwbGF5ZXI6CiAgICAgICAgICAgIGNoWzYsIGN5LCBjeF0gKz0gbG9nX3NoaXBzCiAgICAgICAgICAgIGNoWzgsIGN5LCBjeF0gPSBtYXRoLnNpbihhbmdsZSkKICAgICAgICAgICAgY2hbOSwgY3ksIGN4XSA9IG1hdGguY29zKGFuZ2xlKQogICAgICAgICAgICBteV9zaGlwcyArPSBzaGlwcwogICAgICAgIGVsaWYgb3duZXIgIT0gLTE6CiAgICAgICAgICAgIGNoWzcsIGN5LCBjeF0gKz0gbG9nX3NoaXBzCiAgICAgICAgICAgIGNoWzEwLCBjeSwgY3hdID0gbWF0aC5zaW4oYW5nbGUpCiAgICAgICAgICAgIGNoWzExLCBjeSwgY3hdID0gbWF0aC5jb3MoYW5nbGUpCiAgICAgICAgICAgIGVuZW15X3NoaXBzICs9IHNoaXBzCgogICAgc2NhbGFycyA9IHRvcmNoLnRlbnNvcigKICAgICAgICBbCiAgICAgICAgICAgIHN0ZXAgLyBNQVhfU1RFUFMsCiAgICAgICAgICAgIG1hdGgubG9nMXAobXlfc2hpcHMpIC8gU0hJUFNfTE9HX01BWCwKICAgICAgICAgICAgbWF0aC5sb2cxcChlbmVteV9zaGlwcykgLyBTSElQU19MT0dfTUFYLAogICAgICAgICAgICBteV9wbGFuZXRfY291bnQgLyAxMC4wLAogICAgICAgICAgICBlbmVteV9wbGFuZXRfY291bnQgLyAxMC4wLAogICAgICAgICAgICBhbmd1bGFyX3ZlbG9jaXR5LAogICAgICAgICAgICBsZW4oY29tZXRfaWRzKSAvIDIwLjAsCiAgICAgICAgXSwKICAgICAgICBkdHlwZT10b3JjaC5mbG9hdDMyLAogICAgKQogICAgcmV0dXJuIGNoLCBzY2FsYXJzLCBteV9wbGFuZXRzCgoKY2xhc3MgQ05OdjEobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoTlVNX0NIQU5ORUxTLCAzMiwgMywgcGFkZGluZz0xKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Db252MmQoMzIsIERfTU9ERUwsIDMsIHBhZGRpbmc9MSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uQ29udjJkKERfTU9ERUwsIERfTU9ERUwsIDMsIHBhZGRpbmc9MiwgZGlsYXRpb249MiksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uQ29udjJkKERfTU9ERUwsIERfTU9ERUwsIDMsIHBhZGRpbmc9NCwgZGlsYXRpb249NCksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uQ29udjJkKERfTU9ERUwsIERfTU9ERUwsIDMsIHBhZGRpbmc9OCwgZGlsYXRpb249OCksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICApCiAgICAgICAgc2VsZi5zY2FsYXJfbWxwID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKFNDQUxBUl9ESU0sIDMyKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5MaW5lYXIoMzIsIDMyKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICkKICAgICAgICBzZWxmLmhlYWRfdHJ1bmsgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoRF9NT0RFTCArIDMyLCA5NiksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICApCiAgICAgICAgc2VsZi5sYXVuY2hfaGVhZCA9IG5uLkxpbmVhcig5NiwgMSkKICAgICAgICBzZWxmLnRhcmdldF9oZWFkID0gbm4uTGluZWFyKDk2LCBHUklEICogR1JJRCkKICAgICAgICBzZWxmLnNoaXBfaGVhZCA9IG5uLkxpbmVhcig5NiwgMSkKICAgICAgICBzZWxmLnZhbHVlX2hlYWQgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoRF9NT0RFTCArIDMyLCAzMiksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uTGluZWFyKDMyLCAxKSwKICAgICAgICApCiAgICAgICAgc2VsZi5mcmFjX2xvZ19zdGQgPSBubi5QYXJhbWV0ZXIodG9yY2gubG9nKHRvcmNoLnRlbnNvcigwLjIpKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBjaGFubmVscywgc2NhbGFycyk6CiAgICAgICAgZmVhdF9tYXAgPSBzZWxmLmNvbnYoY2hhbm5lbHMpCiAgICAgICAgc2NhbGFyX2ZlYXQgPSBzZWxmLnNjYWxhcl9tbHAoc2NhbGFycykKICAgICAgICByZXR1cm4gZmVhdF9tYXAsIHNjYWxhcl9mZWF0CgogICAgZGVmIHZhbHVlKHNlbGYsIGZlYXRfbWFwLCBzY2FsYXJfZmVhdCk6CiAgICAgICAgcG9vbGVkID0gZmVhdF9tYXAubWVhbihkaW09KDIsIDMpKQogICAgICAgIHJldHVybiBzZWxmLnZhbHVlX2hlYWQodG9yY2guY2F0KFtwb29sZWQsIHNjYWxhcl9mZWF0XSwgZGltPS0xKSkuc3F1ZWV6ZSgtMSkKCiAgICBkZWYgYWN0KHNlbGYsIGZlYXRfbWFwLCBzY2FsYXJfZmVhdCwgcGxhbmV0X2Nvb3Jkcyk6CiAgICAgICAgbiA9IHBsYW5ldF9jb29yZHMuc2l6ZSgwKQogICAgICAgIGlmIG4gPT0gMDoKICAgICAgICAgICAgZW1wdHkgPSB0b3JjaC5lbXB0eSgwLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgICAgICByZXR1cm4gZW1wdHksIHRvcmNoLmVtcHR5KDAsIEdSSUQgKiBHUklEKSwgZW1wdHkKCiAgICAgICAgZ3JpZCA9IHBsYW5ldF9jb29yZHMuY2xvbmUoKQogICAgICAgIGdyaWRbOiwgMF0gPSAoZ3JpZFs6LCAwXSAvIEJPQVJEX1NJWkUpICogMiAtIDEKICAgICAgICBncmlkWzosIDFdID0gKGdyaWRbOiwgMV0gLyBCT0FSRF9TSVpFKSAqIDIgLSAxCiAgICAgICAgZ3JpZCA9IGdyaWQudmlldygxLCBuLCAxLCAyKQogICAgICAgIHNhbXBsZWQgPSBGLmdyaWRfc2FtcGxlKGZlYXRfbWFwLCBncmlkLCBtb2RlPSJiaWxpbmVhciIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpCiAgICAgICAgcGVyX3BsYW5ldCA9IHNhbXBsZWQuc3F1ZWV6ZSgtMSkuc3F1ZWV6ZSgwKS50cmFuc3Bvc2UoMCwgMSkKCiAgICAgICAgcyA9IHNjYWxhcl9mZWF0LmV4cGFuZChuLCAtMSkKICAgICAgICB4ID0gc2VsZi5oZWFkX3RydW5rKHRvcmNoLmNhdChbcGVyX3BsYW5ldCwgc10sIGRpbT0tMSkpCiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgc2VsZi5sYXVuY2hfaGVhZCh4KS5zcXVlZXplKC0xKSwKICAgICAgICAgICAgc2VsZi50YXJnZXRfaGVhZCh4KSwKICAgICAgICAgICAgc2VsZi5zaGlwX2hlYWQoeCkuc3F1ZWV6ZSgtMSksCiAgICAgICAgKQoKCl9NT0RFTDogQ05OdjEgfCBOb25lID0gTm9uZQoKCmRlZiBfZ2V0X21vZGVsKCkgLT4gQ05OdjE6CiAgICBnbG9iYWwgX01PREVMCiAgICBpZiBfTU9ERUwgaXMgTm9uZToKICAgICAgICBtID0gQ05OdjEoKQogICAgICAgIG0uZXZhbCgpCiAgICAgICAgaWYgV0VJR0hUU19QQVRILmV4aXN0cygpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBtLmxvYWRfc3RhdGVfZGljdCh0b3JjaC5sb2FkKFdFSUdIVFNfUEFUSCwgbWFwX2xvY2F0aW9uPSJjcHUiKSwgc3RyaWN0PUZhbHNlKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICB3YXJuaW5ncy53YXJuKGYiY25uX3YxOiBmYWlsZWQgdG8gbG9hZCB3ZWlnaHRzOiB7ZX0iKQogICAgICAgIF9NT0RFTCA9IG0KICAgIHJldHVybiBfTU9ERUwKCgpkZWYgcmVsb2FkX3dlaWdodHMoKSAtPiBOb25lOgogICAgIiIiSW52YWxpZGF0ZSB0aGUgY2FjaGVkIG1vZGVsIHNvIHRoZSBuZXh0IGNhbGwgcmVsb2FkcyBmcm9tIGRpc2suIiIiCiAgICBnbG9iYWwgX01PREVMCiAgICBfTU9ERUwgPSBOb25lCgoKQHJlZ2lzdGVyKAogICAgImNubl92MSIsCiAgICAiQ29udm9sdXRpb25hbCBwb2xpY3kgb3ZlciBhIDUww5c1MCBwbGF5ZXItcmVsYXRpdmUgcmFzdGVyIOKAlCAiCiAgICAiMTUgY2hhbm5lbHMsIGRpbGF0ZWQgc3RhY2ssIHBlci1wbGFuZXQgYWN0aW9uIGhlYWQuIFVudHJhaW5lZC4iLAopCmRlZiBjbm5fdjFfYWdlbnQob2JzKToKICAgIGNoYW5uZWxzLCBzY2FsYXJzLCBteV9wbGFuZXRzID0gZmVhdHVyaXplKG9icykKICAgIGlmIG5vdCBteV9wbGFuZXRzOgogICAgICAgIHJldHVybiBbXQoKICAgIG1vZGVsID0gX2dldF9tb2RlbCgpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmZWF0X21hcCwgc2NhbGFyX2ZlYXQgPSBtb2RlbChjaGFubmVscy51bnNxdWVlemUoMCksIHNjYWxhcnMudW5zcXVlZXplKDApKQogICAgICAgIGNvb3JkcyA9IHRvcmNoLnRlbnNvcihbW3gsIHldIGZvciAoXywgeCwgeSwgXykgaW4gbXlfcGxhbmV0c10sIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgbGF1bmNoLCB0YXJnZXRzLCBzaGlwX2ZyYWMgPSBtb2RlbC5hY3QoZmVhdF9tYXAsIHNjYWxhcl9mZWF0LCBjb29yZHMpCgogICAgbGF1bmNoX3AgPSB0b3JjaC5zaWdtb2lkKGxhdW5jaCkKICAgIHNoaXBfcCA9IHRvcmNoLnNpZ21vaWQoc2hpcF9mcmFjKQogICAgdGFyZ2V0X2NlbGxzID0gdGFyZ2V0cy5hcmdtYXgoZGltPS0xKQoKICAgIG1vdmVzID0gW10KICAgIGZvciBpLCAocGlkLCB4LCB5LCBzaGlwcykgaW4gZW51bWVyYXRlKG15X3BsYW5ldHMpOgogICAgICAgIGlmIGxhdW5jaF9wW2ldLml0ZW0oKSA8IExBVU5DSF9USFJFU0hPTEQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdGMgPSBpbnQodGFyZ2V0X2NlbGxzW2ldLml0ZW0oKSkKICAgICAgICB0Y3ksIHRjeCA9IGRpdm1vZCh0YywgR1JJRCkKICAgICAgICB0eCA9ICh0Y3ggKyAwLjUpICogQ0VMTAogICAgICAgIHR5ID0gKHRjeSArIDAuNSkgKiBDRUxMCiAgICAgICAgbiA9IG1heCgxLCBtaW4oaW50KHNoaXBzKSwgaW50KHJvdW5kKHNoaXBfcFtpXS5pdGVtKCkgKiBzaGlwcykpKSkKICAgICAgICBpZiBuIDwgMToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBhbmdsZSA9IG1hdGguYXRhbjIodHkgLSB5LCB0eCAtIHgpCiAgICAgICAgbW92ZXMuYXBwZW5kKFtwaWQsIGFuZ2xlLCBuXSkKCiAgICByZXR1cm4gbW92ZXMK',
    'agents/cnn_v1/common.py': 'IiIiU2hhcmVkIGhlbHBlcnMgZm9yIHRoZSBDTk4gdjEgdHJhaW5pbmcgcGlwZWxpbmUuCgpCcmlkZ2VzIHRoZSBlbnYncyBhY3Rpb24gZm9ybWF0IChgW1tmcm9tX3BsYW5ldF9pZCwgYW5nbGUsIG51bV9zaGlwc10sIC4uLl1gKQphbmQgdGhlIHBvbGljeSdzIHBlci1wbGFuZXQgb3V0cHV0IHR1cGxlcyAobGF1bmNoLCB0YXJnZXRfY2VsbCwgc2hpcF9mcmFjKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gLmFnZW50IGltcG9ydCBCT0FSRF9TSVpFLCBDRUxMLCBHUklELCBXRUlHSFRTX1BBVEgsIENOTnYxCgpSQVlfRElTVCA9IDMwLjAgICMgdW5pdHMgYWxvbmcgdGhlIGFuZ2xlIHRoYXQgd2UgdHJlYXQgYXMgdGhlICJ0YXJnZXQiIG9mIGEgbGF1bmNoCgpfUkVQT19ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQucGFyZW50ClRSQUlOX1JPT1QgPSBfUkVQT19ST09UIC8gImxvZ3MiIC8gInRyYWluaW5nIgpXRUlHSFRTX0RJUiA9IFBhdGgoV0VJR0hUU19QQVRIKS5wYXJlbnQKCgpkZWYgYW5nbGVfdG9fY2VsbChweDogZmxvYXQsIHB5OiBmbG9hdCwgYW5nbGU6IGZsb2F0LCBkaXN0OiBmbG9hdCA9IFJBWV9ESVNUKSAtPiBpbnQ6CiAgICAiIiJQcm9qZWN0IGEgcmF5IGZyb20gKHB4LCBweSkgYXQgYGFuZ2xlYCBkaXN0YW5jZSBgZGlzdGAsIHNuYXAgdG8gYSBncmlkIGNlbGwuIiIiCiAgICB0eCA9IG1heCgwLjAsIG1pbihCT0FSRF9TSVpFIC0gMC4wMDEsIHB4ICsgZGlzdCAqIG1hdGguY29zKGFuZ2xlKSkpCiAgICB0eSA9IG1heCgwLjAsIG1pbihCT0FSRF9TSVpFIC0gMC4wMDEsIHB5ICsgZGlzdCAqIG1hdGguc2luKGFuZ2xlKSkpCiAgICBjeCA9IG1pbihHUklEIC0gMSwgaW50KHR4IC8gQ0VMTCkpCiAgICBjeSA9IG1pbihHUklEIC0gMSwgaW50KHR5IC8gQ0VMTCkpCiAgICByZXR1cm4gY3kgKiBHUklEICsgY3gKCgpkZWYgZGVjb2RlX3RlYWNoZXJfYWN0aW9uKG1vdmVzLCBteV9wbGFuZXRzKToKICAgICIiIlR1cm4gYFtbZnJvbV9pZCwgYW5nbGUsIHNoaXBzXSwgLi4uXWAgaW50byBwZXItcGxhbmV0IGxhYmVscy4KCiAgICBSZXR1cm5zIHRocmVlIGxpc3RzIGFsaWduZWQgd2l0aCBgbXlfcGxhbmV0c2A6CiAgICAgIGxhdW5jaGVkW2ldICAgICA6IDEuMCBpZiBwbGFuZXQgaSBoYWQgYW55IGxhdW5jaGVzLCBlbHNlIDAuMAogICAgICB0YXJnZXRfY2VsbFtpXSAgOiBpbnQgaW4gWzAsIEdSSUQqR1JJRCkg4oCUIGZpcnN0IHRhcmdldCBjZWxsIGlmIGxhdW5jaGVkLCBlbHNlIDAKICAgICAgc2hpcF9mcmFjW2ldICAgIDogc3VtX29mX3NoaXBzIC8gZ2Fycmlzb24sIGNsaXBwZWQgdG8gKDAsIDFdLCBlbHNlIDAKCiAgICBGb3IgcGxhbmV0cyB3aXRoIG11bHRpcGxlIGxhdW5jaGVzIGluIGEgdHVybiB3ZSB1c2UgdGhlIGZpcnN0IGxhdW5jaCdzCiAgICB0YXJnZXQgYW5kIHN1bSB0aGUgc2hpcHMg4oCUIGdvb2QgZW5vdWdoIGZvciBhIGJlaGF2aW9yLWNsb25pbmcgd2FybS1zdGFydC4KICAgICIiIgogICAgaWRfdG9faWR4ID0ge3BbMF06IGkgZm9yIGksIHAgaW4gZW51bWVyYXRlKG15X3BsYW5ldHMpfQogICAgbiA9IGxlbihteV9wbGFuZXRzKQogICAgbGF1bmNoZWQgPSBbMC4wXSAqIG4KICAgIHRhcmdldF9jZWxsID0gWzBdICogbgogICAgc2hpcF9mcmFjID0gWzAuMF0gKiBuCiAgICBzaGlwX3N1bSA9IFswXSAqIG4KCiAgICBmb3IgbSBpbiBtb3ZlczoKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShtLCAobGlzdCwgdHVwbGUpKSBvciBsZW4obSkgPCAzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHBpZCwgYW5nbGUsIHNoaXBzID0gbVswXSwgZmxvYXQobVsxXSksIGludChtWzJdKQogICAgICAgIGlmIHBpZCBub3QgaW4gaWRfdG9faWR4OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGkgPSBpZF90b19pZHhbcGlkXQogICAgICAgIF8sIHgsIHksIGdhcnJpc29uID0gbXlfcGxhbmV0c1tpXQogICAgICAgIGlmIGxhdW5jaGVkW2ldID09IDAuMDoKICAgICAgICAgICAgdGFyZ2V0X2NlbGxbaV0gPSBhbmdsZV90b19jZWxsKHgsIHksIGFuZ2xlKQogICAgICAgIGxhdW5jaGVkW2ldID0gMS4wCiAgICAgICAgc2hpcF9zdW1baV0gKz0gc2hpcHMKICAgICAgICBzaGlwX2ZyYWNbaV0gPSBtaW4oMS4wLCBzaGlwX3N1bVtpXSAvIG1heCgxLCBnYXJyaXNvbikpCiAgICByZXR1cm4gbGF1bmNoZWQsIHRhcmdldF9jZWxsLCBzaGlwX2ZyYWMKCgpkZWYgZnJlc2hfbW9kZWwoKSAtPiBDTk52MToKICAgIHJldHVybiBDTk52MSgpCgoKZGVmIHNhdmVfbW9kZWwobW9kZWw6IENOTnYxLCBwYXRoOiBQYXRoIHwgc3RyIHwgTm9uZSA9IE5vbmUpIC0+IFBhdGg6CiAgICBpbXBvcnQgdG9yY2gKCiAgICB0YXJnZXQgPSBQYXRoKHBhdGgpIGlmIHBhdGggZWxzZSBXRUlHSFRTX1BBVEgKICAgIHRhcmdldC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdG9yY2guc2F2ZShtb2RlbC5zdGF0ZV9kaWN0KCksIHRhcmdldCkKICAgIHJldHVybiB0YXJnZXQKCgpkZWYgbG9hZF9tb2RlbChwYXRoOiBQYXRoIHwgc3RyIHwgTm9uZSA9IE5vbmUpIC0+IENOTnYxOgogICAgaW1wb3J0IHRvcmNoCgogICAgc3JjID0gUGF0aChwYXRoKSBpZiBwYXRoIGVsc2UgV0VJR0hUU19QQVRICiAgICBtID0gZnJlc2hfbW9kZWwoKQogICAgaWYgc3JjLmV4aXN0cygpOgogICAgICAgIG0ubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQoc3JjLCBtYXBfbG9jYXRpb249ImNwdSIpLCBzdHJpY3Q9RmFsc2UpCiAgICByZXR1cm4gbQo=',
    'agents/cnn_v1/bc.py': 'IiIiQmVoYXZpb3IgY2xvbmluZyB0cmFpbmVyIOKAlCB3YXJtLXN0YXJ0IHRoZSBDTk4gb24gYSB0ZWFjaGVyIHBvbGljeS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIsIFRlbnNvckRhdGFzZXQKCmZyb20gLmFnZW50IGltcG9ydCBDTk52MQpmcm9tIC5jb21tb24gaW1wb3J0IGZyZXNoX21vZGVsLCBsb2FkX21vZGVsLCBzYXZlX21vZGVsCgoKZGVmIF9wbGFuZXRfZmVhdHVyZXMobW9kZWw6IENOTnYxLCBjaGFubmVsczogdG9yY2guVGVuc29yLCBzY2FsYXJzOiB0b3JjaC5UZW5zb3IsIGNvb3JkczogdG9yY2guVGVuc29yKToKICAgICIiIk1pcnJvciBvZiBDTk52MS5hY3QgYnV0IGJhdGNoZWQuIGNvb3JkcyBpcyAoQiwgTSwgMikuIiIiCiAgICBmZWF0X21hcCA9IG1vZGVsLmNvbnYoY2hhbm5lbHMpCiAgICBzY2FsYXJfZmVhdCA9IG1vZGVsLnNjYWxhcl9tbHAoc2NhbGFycykKICAgIEIsIE0sIF8gPSBjb29yZHMuc2hhcGUKICAgIGdyaWQgPSBjb29yZHMuY2xvbmUoKQogICAgZ3JpZFsuLi4sIDBdID0gKGdyaWRbLi4uLCAwXSAvIDEwMC4wKSAqIDIgLSAxCiAgICBncmlkWy4uLiwgMV0gPSAoZ3JpZFsuLi4sIDFdIC8gMTAwLjApICogMiAtIDEKICAgIGdyaWQgPSBncmlkLnZpZXcoQiwgTSwgMSwgMikKICAgIHNhbXBsZWQgPSBGLmdyaWRfc2FtcGxlKGZlYXRfbWFwLCBncmlkLCBtb2RlPSJiaWxpbmVhciIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpICAjIChCLCBELCBNLCAxKQogICAgcGVyX3BsYW5ldCA9IHNhbXBsZWQuc3F1ZWV6ZSgtMSkucGVybXV0ZSgwLCAyLCAxKSAgIyAoQiwgTSwgRCkKICAgIHNmZWF0ID0gc2NhbGFyX2ZlYXQudW5zcXVlZXplKDEpLmV4cGFuZCgtMSwgTSwgLTEpICAjIChCLCBNLCAzMikKICAgIHggPSBtb2RlbC5oZWFkX3RydW5rKHRvcmNoLmNhdChbcGVyX3BsYW5ldCwgc2ZlYXRdLCBkaW09LTEpKSAgIyAoQiwgTSwgOTYpCiAgICBsYXVuY2ggPSBtb2RlbC5sYXVuY2hfaGVhZCh4KS5zcXVlZXplKC0xKSAgIyAoQiwgTSkKICAgIHRhcmdldCA9IG1vZGVsLnRhcmdldF9oZWFkKHgpICAjIChCLCBNLCBHUklEKkdSSUQpCiAgICBzaGlwID0gbW9kZWwuc2hpcF9oZWFkKHgpLnNxdWVlemUoLTEpICAjIChCLCBNKQogICAgcmV0dXJuIGxhdW5jaCwgdGFyZ2V0LCBzaGlwCgoKZGVmIHRyYWluX2JjKAogICAgZGF0YXNldDogZGljdFtzdHIsIHRvcmNoLlRlbnNvcl0sCiAgICBlcG9jaHM6IGludCA9IDMsCiAgICBiYXRjaF9zaXplOiBpbnQgPSAzMiwKICAgIGxyOiBmbG9hdCA9IDFlLTMsCiAgICBsYXVuY2hfd2VpZ2h0OiBmbG9hdCA9IDEuMCwKICAgIHRhcmdldF93ZWlnaHQ6IGZsb2F0ID0gMS4wLAogICAgZnJhY193ZWlnaHQ6IGZsb2F0ID0gMC41LAogICAgcmVzdW1lX2Zyb206IFBhdGggfCBzdHIgfCBOb25lID0gTm9uZSwKICAgIHNhdmVfdG86IFBhdGggfCBzdHIgfCBOb25lID0gTm9uZSwKICAgIHZlcmJvc2U6IGJvb2wgPSBUcnVlLAopIC0+IENOTnYxOgogICAgbW9kZWwgPSBsb2FkX21vZGVsKHJlc3VtZV9mcm9tKSBpZiByZXN1bWVfZnJvbSBlbHNlIGZyZXNoX21vZGVsKCkKICAgIG1vZGVsLnRyYWluKCkKICAgIG9wdCA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj1scikKCiAgICBkcyA9IFRlbnNvckRhdGFzZXQoCiAgICAgICAgZGF0YXNldFsiY2hhbm5lbHMiXSwKICAgICAgICBkYXRhc2V0WyJzY2FsYXJzIl0sCiAgICAgICAgZGF0YXNldFsiY29vcmRzIl0sCiAgICAgICAgZGF0YXNldFsibGF1bmNoZWQiXSwKICAgICAgICBkYXRhc2V0WyJ0YXJnZXRzIl0sCiAgICAgICAgZGF0YXNldFsiZnJhY3MiXSwKICAgICAgICBkYXRhc2V0WyJtYXNrIl0sCiAgICApCiAgICBsb2FkZXIgPSBEYXRhTG9hZGVyKGRzLCBiYXRjaF9zaXplPWJhdGNoX3NpemUsIHNodWZmbGU9VHJ1ZSwgbnVtX3dvcmtlcnM9MCkKCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoZXBvY2hzKToKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgc3VtcyA9IHsibGF1bmNoIjogMC4wLCAidGFyZ2V0IjogMC4wLCAiZnJhYyI6IDAuMCwgInRvdGFsIjogMC4wLCAibiI6IDB9CiAgICAgICAgZm9yIGJhdGNoIGluIGxvYWRlcjoKICAgICAgICAgICAgY2hhbm5lbHMsIHNjYWxhcnMsIGNvb3JkcywgbGF1bmNoZWQsIHRhcmdldHMsIGZyYWNzLCBtYXNrID0gYmF0Y2gKICAgICAgICAgICAgcHJlZF9sYXVuY2gsIHByZWRfdGFyZ2V0LCBwcmVkX2ZyYWMgPSBfcGxhbmV0X2ZlYXR1cmVzKG1vZGVsLCBjaGFubmVscywgc2NhbGFycywgY29vcmRzKQoKICAgICAgICAgICAgIyBNYXNrIHRvIGxpdmUgcGxhbmV0cwogICAgICAgICAgICBtID0gbWFzay5ib29sKCkKICAgICAgICAgICAgaWYgbm90IG0uYW55KCk6CiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgbGF1bmNoX2xvc3MgPSBGLmJpbmFyeV9jcm9zc19lbnRyb3B5X3dpdGhfbG9naXRzKAogICAgICAgICAgICAgICAgcHJlZF9sYXVuY2hbbV0sIGxhdW5jaGVkW21dLCByZWR1Y3Rpb249Im1lYW4iCiAgICAgICAgICAgICkKCiAgICAgICAgICAgIGxtID0gbSAmIGxhdW5jaGVkLmJvb2woKQogICAgICAgICAgICBpZiBsbS5hbnkoKToKICAgICAgICAgICAgICAgIHRhcmdldF9sb3NzID0gRi5jcm9zc19lbnRyb3B5KHByZWRfdGFyZ2V0W2xtXSwgdGFyZ2V0c1tsbV0sIHJlZHVjdGlvbj0ibWVhbiIpCiAgICAgICAgICAgICAgICBmcmFjX2xvc3MgPSBGLm1zZV9sb3NzKHRvcmNoLnNpZ21vaWQocHJlZF9mcmFjW2xtXSksIGZyYWNzW2xtXSwgcmVkdWN0aW9uPSJtZWFuIikKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHRhcmdldF9sb3NzID0gdG9yY2guemVyb3MoKCksIGRldmljZT1wcmVkX3RhcmdldC5kZXZpY2UpCiAgICAgICAgICAgICAgICBmcmFjX2xvc3MgPSB0b3JjaC56ZXJvcygoKSwgZGV2aWNlPXByZWRfZnJhYy5kZXZpY2UpCgogICAgICAgICAgICBsb3NzID0gKAogICAgICAgICAgICAgICAgbGF1bmNoX3dlaWdodCAqIGxhdW5jaF9sb3NzCiAgICAgICAgICAgICAgICArIHRhcmdldF93ZWlnaHQgKiB0YXJnZXRfbG9zcwogICAgICAgICAgICAgICAgKyBmcmFjX3dlaWdodCAqIGZyYWNfbG9zcwogICAgICAgICAgICApCgogICAgICAgICAgICBvcHQuemVyb19ncmFkKCkKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIHRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIG1heF9ub3JtPTEuMCkKICAgICAgICAgICAgb3B0LnN0ZXAoKQoKICAgICAgICAgICAgc3Vtc1sibGF1bmNoIl0gKz0gbGF1bmNoX2xvc3MuaXRlbSgpCiAgICAgICAgICAgIHN1bXNbInRhcmdldCJdICs9IHRhcmdldF9sb3NzLml0ZW0oKQogICAgICAgICAgICBzdW1zWyJmcmFjIl0gKz0gZnJhY19sb3NzLml0ZW0oKQogICAgICAgICAgICBzdW1zWyJ0b3RhbCJdICs9IGxvc3MuaXRlbSgpCiAgICAgICAgICAgIHN1bXNbIm4iXSArPSAxCgogICAgICAgIGlmIHZlcmJvc2UgYW5kIHN1bXNbIm4iXToKICAgICAgICAgICAgbiA9IHN1bXNbIm4iXQogICAgICAgICAgICBwcmludCgKICAgICAgICAgICAgICAgIGYiZXBvY2gge2Vwb2NoICsgMX0ve2Vwb2Noc30gICIKICAgICAgICAgICAgICAgIGYibGF1bmNoPXtzdW1zWydsYXVuY2gnXS9uOi40Zn0gICIKICAgICAgICAgICAgICAgIGYidGFyZ2V0PXtzdW1zWyd0YXJnZXQnXS9uOi40Zn0gICIKICAgICAgICAgICAgICAgIGYiZnJhYz17c3Vtc1snZnJhYyddL246LjRmfSAgIgogICAgICAgICAgICAgICAgZiJ0b3RhbD17c3Vtc1sndG90YWwnXS9uOi40Zn0gICIKICAgICAgICAgICAgICAgIGYiKHt0aW1lLnRpbWUoKSAtIHQwOi4xZn1zKSIsCiAgICAgICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgICAgICkKCiAgICBtb2RlbC5ldmFsKCkKICAgIGlmIHNhdmVfdG8gaXMgbm90IE5vbmUgb3Igc2F2ZV90byBpcyBOb25lOgogICAgICAgIHBhdGggPSBzYXZlX21vZGVsKG1vZGVsLCBzYXZlX3RvKQogICAgICAgIGlmIHZlcmJvc2U6CiAgICAgICAgICAgIHByaW50KGYic2F2ZWQgd2VpZ2h0czoge3BhdGh9IiwgZmlsZT1zeXMuc3RkZXJyKQogICAgcmV0dXJuIG1vZGVsCg==',
    'agents/cnn_v1/collect.py': 'IiIiQmVoYXZpb3ItY2xvbmluZyBkYXRhc2V0IGNvbGxlY3Rvci4KClJ1bnMgdGVhY2hlci12cy1vcHBvbmVudCBtYXRjaGVzLCBmZWF0dXJpemVzIGV2ZXJ5IG9ic2VydmF0aW9uIGZyb20gdGhlCnRlYWNoZXIncyBQT1YsIGV4dHJhY3RzIHBlci1wbGFuZXQgbGFiZWxzIGZyb20gdGhlIHRlYWNoZXIncyBhY3Rpb25zLCBhbmQKc2F2ZXMgdGhlIHJlc3VsdCBhcyBhIHNpbmdsZSAucHQgZmlsZSBvZiBkaWN0LW9mLXRlbnNvcnMgZm9yIGZhc3QgcmVsb2FkLgoKVmFyaWFibGUgcGxhbmV0IGNvdW50cyBwZXIgc3RlcCBhcmUgaGFuZGxlZCB3aXRoIGEgYm9vbGVhbiBtYXNrIOKAlCBlYWNoCnNhbXBsZSBpcyBwYWRkZWQgdG8gTUFYX1BMQU5FVFMgKHplcm8tZmlsbGVkIGJleW9uZCB0aGUgbGl2ZSBwbGFuZXRzKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgdG9yY2gKCmZyb20gYWdlbnRzIGltcG9ydCBBZ2VudApmcm9tIGthZ2dsZV9lbnZpcm9ubWVudHMgaW1wb3J0IG1ha2UKCmZyb20gLmFnZW50IGltcG9ydCBHUklELCBmZWF0dXJpemUKZnJvbSAuY29tbW9uIGltcG9ydCBUUkFJTl9ST09ULCBkZWNvZGVfdGVhY2hlcl9hY3Rpb24KCk1BWF9QTEFORVRTID0gMTYKCgpkZWYgX29uZV9nYW1lKHRlYWNoZXI6IEFnZW50LCBvcHBvbmVudDogQWdlbnQsIHRlYWNoZXJfc2xvdDogaW50LCBzZWVkOiBpbnQgfCBOb25lKSAtPiBkaWN0OgogICAgY29uZmlnID0geyJzZWVkIjogc2VlZH0gaWYgc2VlZCBpcyBub3QgTm9uZSBlbHNlIHt9CiAgICBlbnYgPSBtYWtlKCJvcmJpdF93YXJzIiwgY29uZmlndXJhdGlvbj1jb25maWcsIGRlYnVnPUZhbHNlKQogICAgcGxheWVycyA9IFt0ZWFjaGVyLmZuLCBvcHBvbmVudC5mbl0gaWYgdGVhY2hlcl9zbG90ID09IDAgZWxzZSBbb3Bwb25lbnQuZm4sIHRlYWNoZXIuZm5dCiAgICBlbnYucnVuKHBsYXllcnMpCgogICAgY2hhbm5lbHNfbGlzdCA9IFtdCiAgICBzY2FsYXJzX2xpc3QgPSBbXQogICAgY29vcmRzX2xpc3QgPSBbXQogICAgbGF1bmNoZWRfbGlzdCA9IFtdCiAgICB0YXJnZXRzX2xpc3QgPSBbXQogICAgZnJhY3NfbGlzdCA9IFtdCiAgICBtYXNrX2xpc3QgPSBbXQoKICAgIGZvciBzdGVwIGluIGVudi5zdGVwc1s6LTFdOgogICAgICAgIHN0YXRlID0gc3RlcFt0ZWFjaGVyX3Nsb3RdCiAgICAgICAgb2JzID0gc3RhdGUub2JzZXJ2YXRpb24KICAgICAgICBhY3Rpb24gPSBzdGF0ZS5hY3Rpb24gb3IgW10KICAgICAgICAjIGFjdGlvbiBpcyBsaXN0IG9mIFtmcm9tX2lkLCBhbmdsZSwgc2hpcHNdIGZyb20gdGhlIHRlYWNoZXIKICAgICAgICBjaCwgc2MsIG15X3BsYW5ldHMgPSBmZWF0dXJpemUob2JzKQogICAgICAgIGlmIG5vdCBteV9wbGFuZXRzOgogICAgICAgICAgICBjb250aW51ZQoKICAgICAgICBsYXVuY2hlZCwgdGFyZ2V0cywgZnJhY3MgPSBkZWNvZGVfdGVhY2hlcl9hY3Rpb24oYWN0aW9uLCBteV9wbGFuZXRzKQogICAgICAgIG4gPSBtaW4oTUFYX1BMQU5FVFMsIGxlbihteV9wbGFuZXRzKSkKICAgICAgICBjb29yZHMgPSB0b3JjaC56ZXJvcyhNQVhfUExBTkVUUywgMikKICAgICAgICBsID0gdG9yY2guemVyb3MoTUFYX1BMQU5FVFMpCiAgICAgICAgdCA9IHRvcmNoLnplcm9zKE1BWF9QTEFORVRTLCBkdHlwZT10b3JjaC5sb25nKQogICAgICAgIGYgPSB0b3JjaC56ZXJvcyhNQVhfUExBTkVUUykKICAgICAgICBtYXNrID0gdG9yY2guemVyb3MoTUFYX1BMQU5FVFMpCgogICAgICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgICAgICBfLCB4LCB5LCBfID0gbXlfcGxhbmV0c1tpXQogICAgICAgICAgICBjb29yZHNbaSwgMF0gPSB4CiAgICAgICAgICAgIGNvb3Jkc1tpLCAxXSA9IHkKICAgICAgICAgICAgbFtpXSA9IGxhdW5jaGVkW2ldCiAgICAgICAgICAgIHRbaV0gPSB0YXJnZXRzW2ldCiAgICAgICAgICAgIGZbaV0gPSBmcmFjc1tpXQogICAgICAgICAgICBtYXNrW2ldID0gMS4wCgogICAgICAgIGNoYW5uZWxzX2xpc3QuYXBwZW5kKGNoKQogICAgICAgIHNjYWxhcnNfbGlzdC5hcHBlbmQoc2MpCiAgICAgICAgY29vcmRzX2xpc3QuYXBwZW5kKGNvb3JkcykKICAgICAgICBsYXVuY2hlZF9saXN0LmFwcGVuZChsKQogICAgICAgIHRhcmdldHNfbGlzdC5hcHBlbmQodCkKICAgICAgICBmcmFjc19saXN0LmFwcGVuZChmKQogICAgICAgIG1hc2tfbGlzdC5hcHBlbmQobWFzaykKCiAgICBpZiBub3QgY2hhbm5lbHNfbGlzdDoKICAgICAgICByZXR1cm4ge30KICAgIHJldHVybiB7CiAgICAgICAgImNoYW5uZWxzIjogdG9yY2guc3RhY2soY2hhbm5lbHNfbGlzdCksCiAgICAgICAgInNjYWxhcnMiOiB0b3JjaC5zdGFjayhzY2FsYXJzX2xpc3QpLAogICAgICAgICJjb29yZHMiOiB0b3JjaC5zdGFjayhjb29yZHNfbGlzdCksCiAgICAgICAgImxhdW5jaGVkIjogdG9yY2guc3RhY2sobGF1bmNoZWRfbGlzdCksCiAgICAgICAgInRhcmdldHMiOiB0b3JjaC5zdGFjayh0YXJnZXRzX2xpc3QpLAogICAgICAgICJmcmFjcyI6IHRvcmNoLnN0YWNrKGZyYWNzX2xpc3QpLAogICAgICAgICJtYXNrIjogdG9yY2guc3RhY2sobWFza19saXN0KSwKICAgIH0KCgpkZWYgY29sbGVjdF9iY19kYXRhc2V0KAogICAgbnVtX2dhbWVzOiBpbnQsCiAgICB0ZWFjaGVyX2lkOiBzdHIgPSAic25pcGVyX3YxIiwKICAgIG9wcG9uZW50X2lkOiBzdHIgPSAic25pcGVyX3YxIiwKICAgIHNhdmVfdG86IFBhdGggfCBzdHIgfCBOb25lID0gTm9uZSwKICAgIHNlZWRfc3RhcnQ6IGludCA9IDAsCiAgICB2ZXJib3NlOiBib29sID0gVHJ1ZSwKKSAtPiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXToKICAgIHRlYWNoZXIgPSBBZ2VudChpZD10ZWFjaGVyX2lkKQogICAgb3Bwb25lbnQgPSBBZ2VudChpZD1vcHBvbmVudF9pZCkKICAgIGFjY3VtOiBkaWN0W3N0ciwgbGlzdFt0b3JjaC5UZW5zb3JdXSA9IHsKICAgICAgICBrOiBbXSBmb3IgayBpbiAoImNoYW5uZWxzIiwgInNjYWxhcnMiLCAiY29vcmRzIiwgImxhdW5jaGVkIiwgInRhcmdldHMiLCAiZnJhY3MiLCAibWFzayIpCiAgICB9CiAgICB0b3RhbF9zYW1wbGVzID0gMAogICAgZm9yIGcgaW4gcmFuZ2UobnVtX2dhbWVzKToKICAgICAgICB0ZWFjaGVyX3Nsb3QgPSBnICUgMiAgIyBhbHRlcm5hdGUgc2VhdHMgZm9yIGJhbGFuY2UKICAgICAgICBzaGFyZCA9IF9vbmVfZ2FtZSh0ZWFjaGVyLCBvcHBvbmVudCwgdGVhY2hlcl9zbG90LCBzZWVkPXNlZWRfc3RhcnQgKyBnKQogICAgICAgIGlmIG5vdCBzaGFyZDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3IgaywgdiBpbiBzaGFyZC5pdGVtcygpOgogICAgICAgICAgICBhY2N1bVtrXS5hcHBlbmQodikKICAgICAgICB0b3RhbF9zYW1wbGVzICs9IHNoYXJkWyJjaGFubmVscyJdLnNpemUoMCkKICAgICAgICBpZiB2ZXJib3NlIGFuZCAoZyArIDEpICUgMTAgPT0gMDoKICAgICAgICAgICAgcHJpbnQoZiIgIGNvbGxlY3RlZCB7ZyArIDF9L3tudW1fZ2FtZXN9IGdhbWVzLCB7dG90YWxfc2FtcGxlc30gc2FtcGxlcyIsIGZpbGU9c3lzLnN0ZGVycikKCiAgICBvdXQgPSB7azogdG9yY2guY2F0KHYsIGRpbT0wKSBmb3IgaywgdiBpbiBhY2N1bS5pdGVtcygpIGlmIHZ9CiAgICBpZiBzYXZlX3RvOgogICAgICAgIHNhdmVfdG8gPSBQYXRoKHNhdmVfdG8pCiAgICAgICAgc2F2ZV90by5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHRvcmNoLnNhdmUob3V0LCBzYXZlX3RvKQogICAgICAgIGlmIHZlcmJvc2U6CiAgICAgICAgICAgIHByaW50KGYic2F2ZWQgQkMgZGF0YXNldDoge3NhdmVfdG99ICh7dG90YWxfc2FtcGxlc30gc2FtcGxlcykiLCBmaWxlPXN5cy5zdGRlcnIpCiAgICByZXR1cm4gb3V0CgoKZGVmIGRlZmF1bHRfZGF0YXNldF9wYXRoKHRlYWNoZXJfaWQ6IHN0ciwgbnVtX2dhbWVzOiBpbnQpIC0+IFBhdGg6CiAgICByZXR1cm4gVFJBSU5fUk9PVCAvICJkYXRhc2V0cyIgLyBmImJjX3t0ZWFjaGVyX2lkfV97bnVtX2dhbWVzfS5wdCIK',
    'agents/cnn_v1/eval.py': 'IiIiVG91cm5hbWVudCBldmFsdWF0b3Ig4oCUIHBsYXkgTiBnYW1lcyBhZ2FpbnN0IGVhY2ggb3Bwb25lbnQsIHJlcG9ydCB3aW4gcmF0ZS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBzeXMKZnJvbSB0eXBpbmcgaW1wb3J0IEl0ZXJhYmxlCgpmcm9tIHV0aWxzIGltcG9ydCBydW5fbWF0Y2gKCkRFRkFVTFRfT1BQT05FTlRTID0gWyJyYW5kb21fdjEiLCAic25pcGVyX3YxIiwgInBoeXNpY2FsX3YyIl0KCgpkZWYgZXZhbHVhdGVfYWdlbnQoCiAgICBhZ2VudF9pZDogc3RyLAogICAgb3Bwb25lbnRzOiBJdGVyYWJsZVtzdHJdID0gREVGQVVMVF9PUFBPTkVOVFMsCiAgICBnYW1lc19wZXI6IGludCA9IDEwLAogICAgc2VlZF9zdGFydDogaW50ID0gMCwKICAgIHZlcmJvc2U6IGJvb2wgPSBUcnVlLAopIC0+IGRpY3Rbc3RyLCBkaWN0W3N0ciwgaW50XV06CiAgICAiIiJQbGF5IGBnYW1lc19wZXJgIGdhbWVzIHZzIGVhY2ggb3Bwb25lbnQsIGFsdGVybmF0aW5nIHNlYXRzLgoKICAgIFJldHVybnM6IGBge29wcG9uZW50OiB7IndpbnMiOiB3LCAibG9zc2VzIjogbCwgImRyYXdzIjogZCwgIndpbl9yYXRlIjogcH19YGAuCiAgICAiIiIKICAgIHJlc3VsdHM6IGRpY3Rbc3RyLCBkaWN0W3N0ciwgaW50XV0gPSB7fQogICAgZm9yIG9wcCBpbiBvcHBvbmVudHM6CiAgICAgICAgd2lucyA9IGxvc3NlcyA9IGRyYXdzID0gMAogICAgICAgIGZvciBnIGluIHJhbmdlKGdhbWVzX3Blcik6CiAgICAgICAgICAgIGFnZW50X3Nsb3QgPSBnICUgMiAgIyBhbHRlcm5hdGUgd2hvIGlzIFAwCiAgICAgICAgICAgIGlkcyA9IFthZ2VudF9pZCwgb3BwXSBpZiBhZ2VudF9zbG90ID09IDAgZWxzZSBbb3BwLCBhZ2VudF9pZF0KICAgICAgICAgICAgciA9IHJ1bl9tYXRjaChpZHMsIHNlZWQ9c2VlZF9zdGFydCArIGcpCiAgICAgICAgICAgIGlmIHIud2lubmVyID09IGFnZW50X3Nsb3Q6CiAgICAgICAgICAgICAgICB3aW5zICs9IDEKICAgICAgICAgICAgZWxpZiByLndpbm5lciA9PSAxIC0gYWdlbnRfc2xvdDoKICAgICAgICAgICAgICAgIGxvc3NlcyArPSAxCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBkcmF3cyArPSAxCiAgICAgICAgdG90YWwgPSBtYXgoMSwgd2lucyArIGxvc3NlcyArIGRyYXdzKQogICAgICAgIHJlYyA9IHsid2lucyI6IHdpbnMsICJsb3NzZXMiOiBsb3NzZXMsICJkcmF3cyI6IGRyYXdzLCAid2luX3JhdGUiOiB3aW5zIC8gdG90YWx9CiAgICAgICAgcmVzdWx0c1tvcHBdID0gcmVjCiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgcHJpbnQoCiAgICAgICAgICAgICAgICBmIiAgdnMge29wcDo8MTR9IHt3aW5zfS17bG9zc2VzfS17ZHJhd3N9ICAiCiAgICAgICAgICAgICAgICBmIih7MTAwICogcmVjWyd3aW5fcmF0ZSddOi4wZn0lKSIsCiAgICAgICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgICAgICkKICAgIHJldHVybiByZXN1bHRzCg==',
    'agents/cnn_v1/ppo.py': '...',
    'utils/__init__.py': 'ZnJvbSAucGFja2VyIGltcG9ydCBwYWNrX2FnZW50CmZyb20gLnJlY29yZGVyIGltcG9ydCByZWNvcmRfbWF0Y2gKZnJvbSAucnVubmVyIGltcG9ydCAoCiAgICBNYXRjaFJlc3VsdCwKICAgIFJFUExBWV9ST09ULAogICAgY29tcHV0ZV9zY29yZXMsCiAgICBtYWtlX3J1bl9pZCwKICAgIHJ1bl9tYXRjaCwKICAgIHNhdmVfcmVwbGF5LAogICAgdHJhaW5fbWF0Y2gsCikKZnJvbSAuc3VibWl0dGVyIGltcG9ydCBzdWJtaXRfYWdlbnQKCl9fYWxsX18gPSBbCiAgICAiTWF0Y2hSZXN1bHQiLAogICAgIlJFUExBWV9ST09UIiwKICAgICJjb21wdXRlX3Njb3JlcyIsCiAgICAibWFrZV9ydW5faWQiLAogICAgInBhY2tfYWdlbnQiLAogICAgInJlY29yZF9tYXRjaCIsCiAgICAicnVuX21hdGNoIiwKICAgICJzYXZlX3JlcGxheSIsCiAgICAic3VibWl0X2FnZW50IiwKICAgICJ0cmFpbl9tYXRjaCIsCl0K',
    'utils/runner.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgYWdlbnRzIGFzIF9hZ2VudHMKCl9SRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudApSRVBMQVlfUk9PVCA9IF9SRVBPX1JPT1QgLyAibG9ncyIgLyAicmVwbGF5cyIKCgpkZWYgbWFrZV9ydW5faWQobW9kZTogc3RyLCBhZ2VudF9pZHM6IGxpc3Rbc3RyXSkgLT4gc3RyOgogICAgIiIiUmV0dXJuIGEgcnVuIGlkIGxpa2UgYGBwbGF5LTIwMjYwNDIxVDAxMzA0NS1zbmlwZXItdnMtcmFuZG9tYGAuIiIiCiAgICB0cyA9IGRhdGV0aW1lLm5vdygpLnN0cmZ0aW1lKCIlWSVtJWRUJUglTSVTIikKICAgIGNvbWJvID0gIi12cy0iLmpvaW4oYWdlbnRfaWRzKQogICAgcmV0dXJuIGYie21vZGV9LXt0c30te2NvbWJvfSIKCgpkZWYgY29tcHV0ZV9zY29yZXMoZW52KSAtPiBsaXN0W2xpc3RbaW50XV06CiAgICAiIiJQZXItc3RlcCB0b3RhbCBzaGlwIGNvdW50IChwbGFuZXRzICsgZmxlZXRzKSBwZXIgcGxheWVyLiIiIgogICAgc2NvcmVzOiBsaXN0W2xpc3RbaW50XV0gPSBbXQogICAgZm9yIHN0ZXAgaW4gZW52LnN0ZXBzOgogICAgICAgIGlmIG5vdCBzdGVwOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG9icyA9IHN0ZXBbMF0ub2JzZXJ2YXRpb24KICAgICAgICBwbGFuZXRzID0gb2JzLmdldCgicGxhbmV0cyIpIG9yIFtdCiAgICAgICAgZmxlZXRzID0gb2JzLmdldCgiZmxlZXRzIikgb3IgW10KICAgICAgICBuID0gbGVuKHN0ZXApCiAgICAgICAgcGVyX3BsYXllciA9IFswXSAqIG4KICAgICAgICBmb3IgcCBpbiBwbGFuZXRzOgogICAgICAgICAgICBvd25lciwgc2hpcHMgPSBwWzFdLCBwWzVdCiAgICAgICAgICAgIGlmIDAgPD0gb3duZXIgPCBuOgogICAgICAgICAgICAgICAgcGVyX3BsYXllcltvd25lcl0gKz0gc2hpcHMKICAgICAgICBmb3IgZiBpbiBmbGVldHM6CiAgICAgICAgICAgIG93bmVyLCBzaGlwcyA9IGZbMV0sIGZbNl0KICAgICAgICAgICAgaWYgMCA8PSBvd25lciA8IG46CiAgICAgICAgICAgICAgICBwZXJfcGxheWVyW293bmVyXSArPSBzaGlwcwogICAgICAgIHNjb3Jlcy5hcHBlbmQocGVyX3BsYXllcikKICAgIHJldHVybiBzY29yZXMKCgpAZGF0YWNsYXNzCmNsYXNzIE1hdGNoUmVzdWx0OgogICAgYWdlbnRfaWRzOiBsaXN0W3N0cl0KICAgIGVudjogQW55CiAgICBzY29yZXM6IGxpc3RbbGlzdFtpbnRdXQogICAgcmV3YXJkczogbGlzdAogICAgd2lubmVyOiBpbnQKCgpkZWYgX3ZhbGlkYXRlX2FuZF9sb2FkKGFnZW50X2lkczogbGlzdFtzdHJdKSAtPiBsaXN0W19hZ2VudHMuQWdlbnRdOgogICAgaWYgbGVuKGFnZW50X2lkcykgbm90IGluICgyLCA0KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYib3JiaXRfd2FycyByZXF1aXJlcyAyIG9yIDQgYWdlbnRzLCBnb3Qge2xlbihhZ2VudF9pZHMpfSIpCiAgICByZXR1cm4gW19hZ2VudHMuQWdlbnQoaWQ9YSkgZm9yIGEgaW4gYWdlbnRfaWRzXQoKCmRlZiBydW5fbWF0Y2goCiAgICBhZ2VudF9pZHM6IGxpc3Rbc3RyXSwKICAgIHNlZWQ6IGludCB8IE5vbmUgPSBOb25lLAogICAgZGVidWc6IGJvb2wgPSBGYWxzZSwKKSAtPiBNYXRjaFJlc3VsdDoKICAgICIiIlBsYXkgb25lIG1hdGNoIGFuZCByZXR1cm4gZW52ICsgc2NvcmVzICsgcmV3YXJkcyArIHdpbm5lci4iIiIKICAgIGZyb20ga2FnZ2xlX2Vudmlyb25tZW50cyBpbXBvcnQgbWFrZQoKICAgIHBsYXllcnMgPSBfdmFsaWRhdGVfYW5kX2xvYWQoYWdlbnRfaWRzKQogICAgY29uZmlnOiBkaWN0ID0geyJzZWVkIjogc2VlZH0gaWYgc2VlZCBpcyBub3QgTm9uZSBlbHNlIHt9CiAgICBlbnYgPSBtYWtlKCJvcmJpdF93YXJzIiwgY29uZmlndXJhdGlvbj1jb25maWcsIGRlYnVnPWRlYnVnKQogICAgZW52LnJ1bihbcC5mbiBmb3IgcCBpbiBwbGF5ZXJzXSkKCiAgICBzY29yZXMgPSBjb21wdXRlX3Njb3JlcyhlbnYpCiAgICBmaW5hbCA9IGVudi5zdGVwc1stMV0KICAgIHJld2FyZHMgPSBbcy5yZXdhcmQgZm9yIHMgaW4gZmluYWxdCiAgICByYW5rZWQgPSBbciBpZiByIGlzIG5vdCBOb25lIGVsc2UgZmxvYXQoIi1pbmYiKSBmb3IgciBpbiByZXdhcmRzXQogICAgd2lubmVyID0gbWF4KHJhbmdlKGxlbihyYW5rZWQpKSwga2V5PWxhbWJkYSBpOiByYW5rZWRbaV0pCgogICAgcmV0dXJuIE1hdGNoUmVzdWx0KAogICAgICAgIGFnZW50X2lkcz1hZ2VudF9pZHMsCiAgICAgICAgZW52PWVudiwKICAgICAgICBzY29yZXM9c2NvcmVzLAogICAgICAgIHJld2FyZHM9cmV3YXJkcywKICAgICAgICB3aW5uZXI9d2lubmVyLAogICAgKQoKCmRlZiBzYXZlX3JlcGxheShlbnYsIHBhdGg6IFBhdGggfCBzdHIpIC0+IFBhdGg6CiAgICAiIiJXcml0ZSBgYGVudi5yZW5kZXIobW9kZT0naHRtbCcpYGAgdG8gYGBwYXRoYGAgYW5kIHJldHVybiBpdC4iIiIKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwYXRoLndyaXRlX3RleHQoZW52LnJlbmRlcihtb2RlPSJodG1sIikpCiAgICByZXR1cm4gcGF0aAoKCmRlZiB0cmFpbl9tYXRjaCgqYXJncywgKiprd2FyZ3MpOgogICAgIiIiUGxhY2Vob2xkZXIg4oCUIHNlbGYtcGxheSB0cmFpbmluZyBsb29wIChub3QgaW1wbGVtZW50ZWQpLiIiIgogICAgcmFpc2UgTm90SW1wbGVtZW50ZWRFcnJvcigidHJhaW5pbmcgbW9kZSBpcyBub3QgaW1wbGVtZW50ZWQgeWV0IikK',
    'utils/packer.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdAppbXBvcnQgaW5zcGVjdAppbXBvcnQgdGFyZmlsZQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBhZ2VudHMKCl9SRVBPX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudApERUZBVUxUX09VVCA9IF9SRVBPX1JPT1QgLyAibG9ncyIgLyAic3VibWlzc2lvbnMiCgoKZGVmIHBhY2tfYWdlbnQoCiAgICBhZ2VudF9pZDogc3RyLAogICAgb3V0X2RpcjogUGF0aCB8IHN0ciB8IE5vbmUgPSBOb25lLAogICAgZXh0cmFfZmlsZXM6IGxpc3RbUGF0aCB8IHN0cl0gfCBOb25lID0gTm9uZSwKICAgIGJ1bmRsZTogYm9vbCA9IEZhbHNlLAopIC0+IGRpY3Rbc3RyLCBQYXRoIHwgTm9uZV06CiAgICAiIiJQYWNrIGEgcmVnaXN0ZXJlZCBhZ2VudCBpbnRvIGEgS2FnZ2xlLXN1Ym1pdHRhYmxlIGBtYWluLnB5YC4KCiAgICBSZXdyaXRlcyB0aGUgYWdlbnQncyBzb3VyY2UgdG8gKDEpIGRyb3AgdGhlIGBmcm9tIC5yZWdpc3RyeSBpbXBvcnQgcmVnaXN0ZXJgCiAgICBpbXBvcnQsICgyKSBzdHJpcCB0aGUgYEByZWdpc3RlciguLi4pYCBkZWNvcmF0b3IsIGFuZCAoMykgYXBwZW5kIGFuCiAgICBgYWdlbnQgPSA8Zm5fbmFtZT5gIGFsaWFzIGF0IHRoZSBlbmQgc28gS2FnZ2xlIGNhbiBwaWNrIHVwIHRoZSBjYWxsYWJsZS4KCiAgICBBcmdzOgogICAgICBhZ2VudF9pZDogcmVnaXN0ZXJlZCBhZ2VudCBpZCAoZS5nLiAic25pcGVyIikuCiAgICAgIG91dF9kaXI6IHdoZXJlIHRvIHdyaXRlIGBtYWluLnB5YC4gRGVmYXVsdHMgdG8gYHN1Ym1pc3Npb25zLzxhZ2VudF9pZD4vYC4KICAgICAgZXh0cmFfZmlsZXM6IHBhdGhzIG9mIGFkZGl0aW9uYWwgZmlsZXMgdG8gY29weSBhbG9uZ3NpZGUgYG1haW4ucHlgIChlLmcuLAogICAgICAgIG1vZGVsIHdlaWdodHMsIGhlbHBlciBtb2R1bGVzKS4gVGhleSdyZSBjb3BpZWQgdmVyYmF0aW0sIGZpbGVuYW1lIHByZXNlcnZlZC4KICAgICAgYnVuZGxlOiBhbHNvIGVtaXQgYSBgLnRhci5nemAgYXQgYG91dF9kaXIvLi4vPGFnZW50X2lkPi50YXIuZ3pgCiAgICAgICAgY29udGFpbmluZyBldmVyeSBmaWxlIGluIGBvdXRfZGlyYC4KCiAgICBSZXR1cm5zOgogICAgICB7Im1haW4iOiBQYXRoIHRvIG1haW4ucHksICJidW5kbGUiOiBQYXRoIG9yIE5vbmUsICJkaXIiOiBvdXRfZGlyfS4KICAgICIiIgogICAgc3BlYyA9IGFnZW50cy5BZ2VudChpZD1hZ2VudF9pZCkKICAgIG91dF9kaXIgPSBQYXRoKG91dF9kaXIpIGlmIG91dF9kaXIgZWxzZSBERUZBVUxUX09VVCAvIGFnZW50X2lkCiAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBtb2R1bGUgPSBpbnNwZWN0LmdldG1vZHVsZShzcGVjLmZuKQogICAgaWYgbW9kdWxlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiY291bGQgbm90IHJlc29sdmUgc291cmNlIG1vZHVsZSBmb3IgYWdlbnQge2FnZW50X2lkIXJ9IikKICAgIHNvdXJjZSA9IGluc3BlY3QuZ2V0c291cmNlKG1vZHVsZSkKCiAgICBtYWluX2NvZGUgPSBfdHJhbnNmb3JtX3NvdXJjZSgKICAgICAgICBzb3VyY2UsCiAgICAgICAgZm5fbmFtZT1zcGVjLmZuLl9fbmFtZV9fLAogICAgICAgIGFnZW50X2lkPWFnZW50X2lkLAogICAgICAgIGRlc2NyaXB0aW9uPXNwZWMuZGVzY3JpcHRpb24sCiAgICApCiAgICBtYWluX3BhdGggPSBvdXRfZGlyIC8gIm1haW4ucHkiCiAgICBtYWluX3BhdGgud3JpdGVfdGV4dChtYWluX2NvZGUpCgogICAgaWYgZXh0cmFfZmlsZXM6CiAgICAgICAgZm9yIGYgaW4gZXh0cmFfZmlsZXM6CiAgICAgICAgICAgIHAgPSBQYXRoKGYpCiAgICAgICAgICAgIChvdXRfZGlyIC8gcC5uYW1lKS53cml0ZV9ieXRlcyhwLnJlYWRfYnl0ZXMoKSkKCiAgICBidW5kbGVfcGF0aDogUGF0aCB8IE5vbmUgPSBOb25lCiAgICBpZiBidW5kbGU6CiAgICAgICAgYnVuZGxlX3BhdGggPSBvdXRfZGlyLnBhcmVudCAvIGYie2FnZW50X2lkfS50YXIuZ3oiCiAgICAgICAgd2l0aCB0YXJmaWxlLm9wZW4oYnVuZGxlX3BhdGgsICJ3Omd6IikgYXMgdGFyOgogICAgICAgICAgICBmb3IgcCBpbiBzb3J0ZWQob3V0X2Rpci5pdGVyZGlyKCkpOgogICAgICAgICAgICAgICAgdGFyLmFkZChwLCBhcmNuYW1lPXAubmFtZSkKCiAgICByZXR1cm4geyJtYWluIjogbWFpbl9wYXRoLCAiYnVuZGxlIjogYnVuZGxlX3BhdGgsICJkaXIiOiBvdXRfZGlyfQoKCmRlZiBfdHJhbnNmb3JtX3NvdXJjZShzb3VyY2U6IHN0ciwgZm5fbmFtZTogc3RyLCBhZ2VudF9pZDogc3RyLCBkZXNjcmlwdGlvbjogc3RyKSAtPiBzdHI6CiAgICB0cmVlID0gYXN0LnBhcnNlKHNvdXJjZSkKICAgIG5ld19ib2R5OiBsaXN0W2FzdC5zdG10XSA9IFtdCiAgICBmb3Igbm9kZSBpbiB0cmVlLmJvZHk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShub2RlLCBhc3QuSW1wb3J0RnJvbSkgYW5kIChub2RlLm1vZHVsZSBvciAiIikuc3BsaXQoIi4iKVstMV0gPT0gInJlZ2lzdHJ5IjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBpc2luc3RhbmNlKG5vZGUsIGFzdC5GdW5jdGlvbkRlZik6CiAgICAgICAgICAgIG5vZGUuZGVjb3JhdG9yX2xpc3QgPSBbCiAgICAgICAgICAgICAgICBkIGZvciBkIGluIG5vZGUuZGVjb3JhdG9yX2xpc3QgaWYgbm90IF9pc19yZWdpc3Rlcl9kZWNvcmF0b3IoZCkKICAgICAgICAgICAgXQogICAgICAgIG5ld19ib2R5LmFwcGVuZChub2RlKQoKICAgIG5ld19ib2R5LmFwcGVuZCgKICAgICAgICBhc3QuQXNzaWduKAogICAgICAgICAgICB0YXJnZXRzPVthc3QuTmFtZShpZD0iYWdlbnQiLCBjdHg9YXN0LlN0b3JlKCkpXSwKICAgICAgICAgICAgdmFsdWU9YXN0Lk5hbWUoaWQ9Zm5fbmFtZSwgY3R4PWFzdC5Mb2FkKCkpLAogICAgICAgICkKICAgICkKICAgIHRyZWUuYm9keSA9IG5ld19ib2R5CiAgICBhc3QuZml4X21pc3NpbmdfbG9jYXRpb25zKHRyZWUpCgogICAgaGVhZGVyID0gKAogICAgICAgICIjIEF1dG8tcGFja2VkIGJ5IHV0aWxzLnBhY2tfYWdlbnRcbiIKICAgICAgICBmIiMgYWdlbnQgaWQgICA6IHthZ2VudF9pZH1cbiIKICAgICAgICBmIiMgZGVzY3JpcHRpb246IHtkZXNjcmlwdGlvbn1cbiIKICAgICAgICBmIiMgcGFja2VkIGF0ICA6IHtkYXRldGltZS5ub3coKS5pc29mb3JtYXQodGltZXNwZWM9J3NlY29uZHMnKX1cblxuIgogICAgKQogICAgcmV0dXJuIGhlYWRlciArIGFzdC51bnBhcnNlKHRyZWUpICsgIlxuIgoKCmRlZiBfaXNfcmVnaXN0ZXJfZGVjb3JhdG9yKGQ6IGFzdC5leHByKSAtPiBib29sOgogICAgaWYgaXNpbnN0YW5jZShkLCBhc3QuQ2FsbCk6CiAgICAgICAgcmV0dXJuIF9pc19yZWdpc3Rlcl9kZWNvcmF0b3IoZC5mdW5jKQogICAgaWYgaXNpbnN0YW5jZShkLCBhc3QuTmFtZSk6CiAgICAgICAgcmV0dXJuIGQuaWQgPT0gInJlZ2lzdGVyIgogICAgaWYgaXNpbnN0YW5jZShkLCBhc3QuQXR0cmlidXRlKToKICAgICAgICByZXR1cm4gZC5hdHRyID09ICJyZWdpc3RlciIKICAgIHJldHVybiBGYWxzZQo=',
    'utils/recorder.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNzdgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKCmRlZiByZWNvcmRfbWF0Y2goCiAgICBvdXRfZGlyOiBQYXRoLAogICAgYWdlbnRfaWRzOiBsaXN0W3N0cl0sCiAgICBlbnY6IEFueSwKICAgIHNjb3JlczogbGlzdFtsaXN0W2ludF1dLAogICAgcmV3YXJkczogbGlzdCwKICAgIHdpbm5lcjogaW50LAopIC0+IGRpY3Rbc3RyLCBQYXRoXToKICAgICIiIldyaXRlIHBlci1tYXRjaCBhcnRpZmFjdHMgdG8gb3V0X2Rpci4KCiAgICBQcm9kdWNlczoKICAgICAgbW92ZXMuY3N2ICAg4oCUIG9uZSByb3cgcGVyIGZsZWV0IGxhdW5jaDogc3RlcCwgcGxheWVyLCBmcm9tX3BsYW5ldCwgYW5nbGUsIHNoaXBzCiAgICAgIHNjb3Jlcy5jc3YgIOKAlCB3aWRlIGZvcm1hdDogc3RlcCwgcGxheWVyXzAsIHBsYXllcl8xLCAuLi4KICAgICAgbWF0Y2guanNvbiAg4oCUIG1hdGNoLWxldmVsIG1ldGFkYXRhIChhZ2VudHMsIHJld2FyZHMsIHdpbm5lciwgbnVtX3N0ZXBzKQogICAgIiIiCiAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGhzOiBkaWN0W3N0ciwgUGF0aF0gPSB7fQoKICAgIG1vdmVzX3BhdGggPSBvdXRfZGlyIC8gIm1vdmVzLmNzdiIKICAgIHdpdGggbW92ZXNfcGF0aC5vcGVuKCJ3IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICB3ID0gY3N2LndyaXRlcihmKQogICAgICAgIHcud3JpdGVyb3coWyJzdGVwIiwgInBsYXllciIsICJmcm9tX3BsYW5ldCIsICJhbmdsZSIsICJzaGlwcyJdKQogICAgICAgIGZvciB0LCBzdGVwIGluIGVudW1lcmF0ZShlbnYuc3RlcHMpOgogICAgICAgICAgICBmb3IgaSwgc3RhdGUgaW4gZW51bWVyYXRlKHN0ZXApOgogICAgICAgICAgICAgICAgYWN0aW9uID0gZ2V0YXR0cihzdGF0ZSwgImFjdGlvbiIsIE5vbmUpIG9yIFtdCiAgICAgICAgICAgICAgICBmb3IgbSBpbiBhY3Rpb246CiAgICAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShtLCAobGlzdCwgdHVwbGUpKSBhbmQgbGVuKG0pID49IDM6CiAgICAgICAgICAgICAgICAgICAgICAgIHcud3JpdGVyb3coW3QsIGksIG1bMF0sIG1bMV0sIG1bMl1dKQogICAgcGF0aHNbIm1vdmVzIl0gPSBtb3Zlc19wYXRoCgogICAgc2NvcmVzX3BhdGggPSBvdXRfZGlyIC8gInNjb3Jlcy5jc3YiCiAgICBuID0gbGVuKHNjb3Jlc1swXSkgaWYgc2NvcmVzIGVsc2UgMAogICAgd2l0aCBzY29yZXNfcGF0aC5vcGVuKCJ3IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICB3ID0gY3N2LndyaXRlcihmKQogICAgICAgIHcud3JpdGVyb3coWyJzdGVwIl0gKyBbZiJwbGF5ZXJfe2l9IiBmb3IgaSBpbiByYW5nZShuKV0pCiAgICAgICAgZm9yIHQsIHJvdyBpbiBlbnVtZXJhdGUoc2NvcmVzKToKICAgICAgICAgICAgdy53cml0ZXJvdyhbdF0gKyBsaXN0KHJvdykpCiAgICBwYXRoc1sic2NvcmVzIl0gPSBzY29yZXNfcGF0aAoKICAgIG1ldGFfcGF0aCA9IG91dF9kaXIgLyAibWF0Y2guanNvbiIKICAgIG1ldGFfcGF0aC53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJhZ2VudHMiOiBhZ2VudF9pZHMsCiAgICAgICAgICAgICAgICAicmV3YXJkcyI6IHJld2FyZHMsCiAgICAgICAgICAgICAgICAid2lubmVyIjogd2lubmVyLAogICAgICAgICAgICAgICAgIm51bV9zdGVwcyI6IGxlbihzY29yZXMpLAogICAgICAgICAgICB9LAogICAgICAgICAgICBpbmRlbnQ9MiwKICAgICAgICApCiAgICApCiAgICBwYXRoc1sibWF0Y2giXSA9IG1ldGFfcGF0aAoKICAgIHJldHVybiBwYXRocwo=',
    'utils/submitter.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gLnBhY2tlciBpbXBvcnQgcGFja19hZ2VudAoKREVGQVVMVF9DT01QRVRJVElPTiA9ICJvcmJpdC13YXJzIgoKCmRlZiBzdWJtaXRfYWdlbnQoCiAgICBhZ2VudF9pZDogc3RyLAogICAgbm90ZTogc3RyID0gIiIsCiAgICBidW5kbGU6IGJvb2wgPSBGYWxzZSwKICAgIGRyeV9ydW46IGJvb2wgPSBGYWxzZSwKICAgIGNvbXBldGl0aW9uOiBzdHIgPSBERUZBVUxUX0NPTVBFVElUSU9OLAogICAga2FnZ2xlX2Jpbjogc3RyIHwgTm9uZSA9IE5vbmUsCikgLT4gZGljdDoKICAgICIiIlBhY2sgYSByZWdpc3RlcmVkIGFnZW50IGFuZCBzdWJtaXQgaXQgdG8gS2FnZ2xlLgoKICAgIE1lc3NhZ2UgY29udmVudGlvbjoKICAgICAgICAiPGFnZW50X2lkPiBAIDxnaXRfc2hhPlstZGlydHldWyDigJQgPG5vdGU+XSIKCiAgICBFeGFtcGxlczoKICAgICAgICAic25pcGVyIEAgYWJjMTIzNCIKICAgICAgICAic25pcGVyIEAgYWJjMTIzNC1kaXJ0eSIKICAgICAgICAic25pcGVyIEAgYWJjMTIzNCDigJQgYnVtcGVkIG1pbi1zaGlwcyB0byAzMCIKCiAgICBSYXRpb25hbGU6CiAgICAgIC0gYGFnZW50X2lkYCBpZGVudGlmaWVzIHdoaWNoIGFnZW50IG9uIHRoZSBsZWFkZXJib2FyZC4KICAgICAgLSBgZ2l0X3NoYWAgbWFrZXMgc3VibWlzc2lvbnMgcmVwcm9kdWNpYmxlIChjaGVjayBvdXQgdGhpcyBTSEEgdG8KICAgICAgICByZXByb2R1Y2UgdGhlIHBhY2tlZCBhcnRpZmFjdCkuCiAgICAgIC0gYC1kaXJ0eWAgd2FybnMgdGhhdCB1bmNvbW1pdHRlZCBjaGFuZ2VzIHdlcmUgaW5jbHVkZWQsIHNvIHRoZQogICAgICAgIGFydGlmYWN0IGNhbm5vdCBiZSByZWNvbnN0cnVjdGVkIGZyb20gZ2l0IGFsb25lLgogICAgICAtIGBub3RlYCBpcyBvcHRpb25hbCBodW1hbi1yZWFkYWJsZSBjb250ZXh0ICh3aGF0J3MgbmV3IC8gd2hhdAogICAgICAgIGNoYW5nZWQgc2luY2UgdGhlIGxhc3Qgc3VibWlzc2lvbikuCgogICAgQXJnczoKICAgICAgYWdlbnRfaWQ6IHJlZ2lzdGVyZWQgYWdlbnQgaWQgKGUuZy4gInNuaXBlciIpLgogICAgICBub3RlOiBmcmVlLXRleHQgbm90ZSBhcHBlbmRlZCBhZnRlciBhbiBlbSBkYXNoLgogICAgICBidW5kbGU6IGlmIFRydWUsIHN1Ym1pdCBgc3VibWlzc2lvbnMvPGFnZW50X2lkPi50YXIuZ3pgOyBlbHNlIHN1Ym1pdAogICAgICAgIHRoZSBiYXJlIGBtYWluLnB5YC4KICAgICAgZHJ5X3J1bjogaWYgVHJ1ZSwgcHJpbnQgdGhlIGthZ2dsZSBjb21tYW5kIHdpdGhvdXQgZXhlY3V0aW5nLgogICAgICBjb21wZXRpdGlvbjoga2FnZ2xlIGNvbXBldGl0aW9uIHNsdWcuCiAgICAgIGthZ2dsZV9iaW46IG92ZXJyaWRlIHBhdGggdG8gdGhlIGthZ2dsZSBDTEkuIEZhbGxzIGJhY2sgdG8KICAgICAgICBgJEtBR0dMRV9CSU5gLCB0aGVuIGB3aGljaCBrYWdnbGVgLgoKICAgIFJldHVybnM6CiAgICAgIERpY3Qgd2l0aCBgc3VibWl0dGVkYCAoYm9vbCksIGBtZXNzYWdlYCwgYGZpbGVgLCBhbmQgKHdoZW4gc3VibWl0dGVkKQogICAgICBgc3Rkb3V0YCBmcm9tIHRoZSBrYWdnbGUgQ0xJLgoKICAgIFJhaXNlczoKICAgICAgUnVudGltZUVycm9yIGlmIGthZ2dsZSBDTEkgaXMgbWlzc2luZywgY3JlZHMgYXJlIG1pc3NpbmcsIG9yIHRoZQogICAgICBzdWJtaXQgY29tbWFuZCBleGl0cyBub24temVyby4KICAgICIiIgogICAgcGFja19yZXN1bHQgPSBwYWNrX2FnZW50KGFnZW50X2lkLCBidW5kbGU9YnVuZGxlKQogICAgc3VibWl0X3BhdGg6IFBhdGggPSBwYWNrX3Jlc3VsdFsiYnVuZGxlIl0gaWYgYnVuZGxlIGVsc2UgcGFja19yZXN1bHRbIm1haW4iXSAgIyB0eXBlOiBpZ25vcmVbYXNzaWdubWVudF0KCiAgICBzaGEgPSBfZ2l0X3NoYSgpCiAgICBkaXJ0eSA9IF9naXRfaXNfZGlydHkoKQogICAgbWVzc2FnZSA9IF9idWlsZF9tZXNzYWdlKGFnZW50X2lkLCBzaGEsIGRpcnR5LCBub3RlKQoKICAgIGthZ2dsZV9iaW4gPSBrYWdnbGVfYmluIG9yIG9zLmVudmlyb24uZ2V0KCJLQUdHTEVfQklOIikgb3Igc2h1dGlsLndoaWNoKCJrYWdnbGUiKQogICAgaWYgbm90IGthZ2dsZV9iaW46CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAia2FnZ2xlIENMSSBub3QgZm91bmQuIEluc3RhbGwgaXQgKGBwaXAgaW5zdGFsbCAtLXVzZXIga2FnZ2xlYCkgb3IgIgogICAgICAgICAgICAic2V0IEtBR0dMRV9CSU4gdG8gaXRzIHBhdGguIgogICAgICAgICkKCiAgICBjbWQgPSBbCiAgICAgICAga2FnZ2xlX2JpbiwKICAgICAgICAiY29tcGV0aXRpb25zIiwKICAgICAgICAic3VibWl0IiwKICAgICAgICBjb21wZXRpdGlvbiwKICAgICAgICAiLWYiLAogICAgICAgIHN0cihzdWJtaXRfcGF0aCksCiAgICAgICAgIi1tIiwKICAgICAgICBtZXNzYWdlLAogICAgXQoKICAgIHByaW50KGYiW3N1Ym1pdF0gZmlsZTogICAge3N1Ym1pdF9wYXRofSIsIGZpbGU9c3lzLnN0ZGVycikKICAgIHByaW50KGYiW3N1Ym1pdF0gbWVzc2FnZToge21lc3NhZ2V9IiwgZmlsZT1zeXMuc3RkZXJyKQogICAgaWYgZHJ5X3J1bjoKICAgICAgICBwcmludChmIltzdWJtaXRdIGRyeSBydW46IHsnICcuam9pbihjbWQpfSIsIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAic3VibWl0dGVkIjogRmFsc2UsCiAgICAgICAgICAgICJtZXNzYWdlIjogbWVzc2FnZSwKICAgICAgICAgICAgImZpbGUiOiBzdHIoc3VibWl0X3BhdGgpLAogICAgICAgICAgICAiY21kIjogY21kLAogICAgICAgIH0KCiAgICBfY2hlY2tfY3JlZHMoKQoKICAgIHJlc3VsdCA9IHN1YnByb2Nlc3MucnVuKGNtZCwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQogICAgaWYgcmVzdWx0LnJldHVybmNvZGUgIT0gMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICJrYWdnbGUgc3VibWl0IGZhaWxlZDpcbiIKICAgICAgICAgICAgZiJzdGRvdXQ6IHtyZXN1bHQuc3Rkb3V0fVxuc3RkZXJyOiB7cmVzdWx0LnN0ZGVycn0iCiAgICAgICAgKQogICAgc3Rkb3V0ID0gcmVzdWx0LnN0ZG91dC5zdHJpcCgpCiAgICBwcmludChmIltzdWJtaXRdIGthZ2dsZToge3N0ZG91dH0iLCBmaWxlPXN5cy5zdGRlcnIpCiAgICByZXR1cm4gewogICAgICAgICJzdWJtaXR0ZWQiOiBUcnVlLAogICAgICAgICJtZXNzYWdlIjogbWVzc2FnZSwKICAgICAgICAiZmlsZSI6IHN0cihzdWJtaXRfcGF0aCksCiAgICAgICAgInN0ZG91dCI6IHN0ZG91dCwKICAgIH0KCgpkZWYgX2J1aWxkX21lc3NhZ2UoYWdlbnRfaWQ6IHN0ciwgc2hhOiBzdHIsIGRpcnR5OiBib29sLCBub3RlOiBzdHIpIC0+IHN0cjoKICAgIHBhcnRzID0gW2FnZW50X2lkLCAiQCIsIHNoYSArICgiLWRpcnR5IiBpZiBkaXJ0eSBlbHNlICIiKV0KICAgIGlmIG5vdGU6CiAgICAgICAgcGFydHMuZXh0ZW5kKFsi4oCUIiwgbm90ZV0pCiAgICByZXR1cm4gIiAiLmpvaW4ocGFydHMpCgoKZGVmIF9naXRfc2hhKCkgLT4gc3RyOgogICAgdHJ5OgogICAgICAgIHIgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWyJnaXQiLCAicmV2LXBhcnNlIiwgIi0tc2hvcnQiLCAiSEVBRCJdLAogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgICAgIGNoZWNrPVRydWUsCiAgICAgICAgKQogICAgICAgIHJldHVybiByLnN0ZG91dC5zdHJpcCgpCiAgICBleGNlcHQgKHN1YnByb2Nlc3MuQ2FsbGVkUHJvY2Vzc0Vycm9yLCBGaWxlTm90Rm91bmRFcnJvcik6CiAgICAgICAgcmV0dXJuICJub2dpdCIKCgpkZWYgX2dpdF9pc19kaXJ0eSgpIC0+IGJvb2w6CiAgICB0cnk6CiAgICAgICAgciA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICBbImdpdCIsICJzdGF0dXMiLCAiLS1wb3JjZWxhaW4iXSwKICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgdGV4dD1UcnVlLAogICAgICAgICAgICBjaGVjaz1UcnVlLAogICAgICAgICkKICAgICAgICByZXR1cm4gYm9vbChyLnN0ZG91dC5zdHJpcCgpKQogICAgZXhjZXB0IChzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvciwgRmlsZU5vdEZvdW5kRXJyb3IpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmRlZiBfY2hlY2tfY3JlZHMoKSAtPiBOb25lOgogICAgaWYgIktBR0dMRV9VU0VSTkFNRSIgaW4gb3MuZW52aXJvbiBhbmQgKAogICAgICAgICJLQUdHTEVfS0VZIiBpbiBvcy5lbnZpcm9uIG9yICJLQUdHTEVfQVBJX0tFWSIgaW4gb3MuZW52aXJvbgogICAgKToKICAgICAgICBpZiAiS0FHR0xFX0tFWSIgbm90IGluIG9zLmVudmlyb24gYW5kICJLQUdHTEVfQVBJX0tFWSIgaW4gb3MuZW52aXJvbjoKICAgICAgICAgICAgb3MuZW52aXJvblsiS0FHR0xFX0tFWSJdID0gb3MuZW52aXJvblsiS0FHR0xFX0FQSV9LRVkiXQogICAgICAgIHJldHVybgogICAga2ogPSBQYXRoLmhvbWUoKSAvICIua2FnZ2xlIiAvICJrYWdnbGUuanNvbiIKICAgIGlmIGtqLmV4aXN0cygpOgogICAgICAgIHJldHVybgogICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICJLYWdnbGUgY3JlZGVudGlhbHMgbm90IGZvdW5kLiBTZXQgS0FHR0xFX1VTRVJOQU1FICsgS0FHR0xFX0tFWSBlbnYgIgogICAgICAgICJ2YXJzIG9yIHBvcHVsYXRlIH4vLmthZ2dsZS9rYWdnbGUuanNvbi4iCiAgICApCg==',
    'agents/__init__.py': 'ZnJvbSAucmVnaXN0cnkgaW1wb3J0IEFnZW50LCBBZ2VudFNwZWMsIGxpc3RfYWdlbnRfc3BlY3MsIGxpc3RfYWdlbnRzLCByZWdpc3RlcgoKZnJvbSAuIGltcG9ydCBwaHlzaWNhbF92MiAgIyBub3FhOiBGNDAxICDigJQgZXZhbCBvcHBvbmVudApmcm9tIC4gaW1wb3J0IGNubl92MSAgICAgICAgIyBub3FhOiBGNDAxICDigJQgdGhlIHRyYWluZWQgYWdlbnQKCl9fYWxsX18gPSBbIkFnZW50IiwgIkFnZW50U3BlYyIsICJsaXN0X2FnZW50cyIsICJsaXN0X2FnZW50X3NwZWNzIiwgInJlZ2lzdGVyIl0K',
}

for rel, b64 in FILES.items():
    p = Path(rel)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))

print('wrote', len(FILES), 'source files to', BASE)


## Verify environment

In [ ]:
import sys
sys.path.insert(0, '/content/orbit-wars')

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

from agents import list_agents
print('bundled agents:', list_agents())


## Train

In [ ]:
import time
from agents.cnn_v1 import train_ppo

t0 = time.time()
train_ppo(
    iterations=200,
    episodes_per_iter=32,
    snapshot_every=10,
    snapshot_pool_size=8,
    self_play_ratio=0.3,
    eval_every=10,
    eval_games=20,
    eval_opponent='physical_v2',
    use_shaping=True,
    episode_progress=True,
    replay_every=10,
    device='cuda',        # force GPU; will raise if cuda unavailable
    verbose=True,
)
print(f'total: {time.time() - t0:.0f}s')


## Save artifacts

In [ ]:
# Copy artifacts to Google Drive.
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

dst = Path('/content/drive/MyDrive/orbit-wars')
dst.mkdir(parents=True, exist_ok=True)

for w in [Path('agents/cnn_v1/weights/cnn_v1.pt'), Path('agents/cnn_v1/weights/latest.pt')]:
    if w.exists():
        shutil.copy(w, dst / w.name)

for jsonl in sorted(Path('logs/training').glob('ppo_*.jsonl')):
    shutil.copy(jsonl, dst / jsonl.name)

replays_src = Path('logs/replays/training')
if replays_src.exists():
    for run_dir in replays_src.iterdir():
        out = dst / 'replays' / run_dir.name
        out.mkdir(parents=True, exist_ok=True)
        for html in run_dir.glob('*.html'):
            shutil.copy(html, out / html.name)

print('copied artifacts to', dst)


In [ ]:
# OR: download artifacts directly to your local machine.
from google.colab import files
import glob
for w in ['agents/cnn_v1/weights/cnn_v1.pt', 'agents/cnn_v1/weights/latest.pt']:
    files.download(w)
for jsonl in sorted(glob.glob('logs/training/ppo_*.jsonl')):
    files.download(jsonl)
for html in sorted(glob.glob('logs/replays/training/*/*.html')):
    files.download(html)
